# ATM Full-Year 2024 — 2-Day Cash Forecasting

**Local Jupyter version**

### Setup
1. Put these files in the **same folder** as this notebook:
   - `ATM_Featured_Dataset_JAN24_JUN24.csv`
   - `ATM_Featured_Dataset_JUL24_DEC24.csv`
   - (optional) `ATM_Event_Calendar.csv`
2. Install packages once (next cell).
3. Run all cells top to bottom.

Outputs written to the same folder:
- `ATM_2Day_Forecast_and_Replenishment_Plan.csv`
- `ATM_Day1_Model_Benchmark.csv` / `ATM_Day2_Model_Benchmark.csv`
- `ATM_2Day_Forecasting_Package.pkl`
- `ATM_2Day_Cash_Dashboard.html` ← open this in your browser


In [ ]:
# Install required packages (run once)
!pip -q install lightgbm xgboost catboost openpyxl joblib scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import os
import math
import json
import warnings
import webbrowser
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_DIR = "."
OUTPUT_DIR = "."
os.makedirs(OUTPUT_DIR, exist_ok=True)


# -------------------- CANONICAL ATM CASH POLICY --------------------
ATM_CAPACITY = 4_000_000          # every ATM holds at most this much cash
ATM_MAX_CAPACITY = ATM_CAPACITY
CASH_UNIT = 500
REORDER_THRESHOLD_PCT = 20        # <= 20% of capacity -> "Refill Now"
MEDIUM_CASH_THRESHOLD_PCT = 40    # <= 40% -> "Refill Soon"
SAFETY_BUFFER_MIN_PCT = 0.10
SAFETY_BUFFER_MAX_PCT = 0.30
assert ATM_CAPACITY > 0 and CASH_UNIT > 0 and ATM_CAPACITY % CASH_UNIT == 0


In [ ]:
file_name_h1 = os.path.join(DATA_DIR, "ATM_Featured_Dataset_JAN24_JUN24.csv")
file_name_h2 = os.path.join(DATA_DIR, "ATM_Featured_Dataset_JUL24_DEC24.csv")

for fp in (file_name_h1, file_name_h2):
    if not os.path.exists(fp):
        raise FileNotFoundError(
            f"Could not find {fp}. Update DATA_DIR in the previous cell "
            f"or place the CSV in this notebook's working directory."
        )

print("Found file:", file_name_h1)
print("Found file:", file_name_h2)


In [ ]:
df_h1 = pd.read_csv(file_name_h1)
df_h2 = pd.read_csv(file_name_h2)

print("Jan\u2013Jun 2024 shape:", df_h1.shape)
print("Jul\u2013Dec 2024 shape:", df_h2.shape)

# Both halves must share the same columns before it's safe to concatenate them
cols_h1, cols_h2 = set(df_h1.columns), set(df_h2.columns)
if cols_h1 != cols_h2:
    raise ValueError(
        "Column mismatch between the two files.\n"
        f"Only in Jan-Jun: {cols_h1 - cols_h2}\n"
        f"Only in Jul-Dec: {cols_h2 - cols_h1}"
    )

df = pd.concat([df_h1, df_h2], ignore_index=True)
df["ATMID"] = df["ATMID"].astype(str).str.strip()   # source IDs are space-padded (e.g. "ABBAGR01    ")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

rows_before_dedup = len(df)
df = (
    df.drop_duplicates(subset=["ATMID", "Date"], keep="first")
      .sort_values(["ATMID", "Date"])
      .reset_index(drop=True)
)
rows_after_dedup = len(df)

print("\nCombined shape (before dedup):", (rows_before_dedup, df.shape[1]))
print("Combined shape (after dedup): ", df.shape)
if rows_before_dedup != rows_after_dedup:
    print(f"WARNING: removed {rows_before_dedup - rows_after_dedup} overlapping ATM/Date "
          f"row(s) found in both files (kept the Jan\u2013Jun version).")
else:
    print("No overlapping ATM/Date rows between the two files \u2014 clean full-year join.")

print("\nCombined date range:", df["Date"].min().date(), "to", df["Date"].max().date())
_missing_days = pd.date_range(df["Date"].min(), df["Date"].max()).difference(pd.DatetimeIndex(df["Date"].unique()))
print(f"Calendar days with NO data: {len(_missing_days)} "
      f"({_missing_days.min().date() if len(_missing_days) else '-'} to {_missing_days.max().date() if len(_missing_days) else '-'})")
display(df.head())


In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))


**Convert date and validate dataset**

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

if df["Date"].isna().any():
    print("WARNING: Invalid dates found:", df["Date"].isna().sum())
else:
    print("All dates converted successfully.")

print("\nDate range:")
print(df["Date"].min(), "to", df["Date"].max())

print("\nNumber of unique dates:")
print(df["Date"].nunique())

print("\nNumber of unique ATMs:")
print(df["ATMID"].nunique())

print("\nATM observations:")
display(df.groupby("ATMID").size().describe())


In [ ]:
date_count_per_atm = (
    df.groupby("ATMID")["Date"]
      .nunique()
)

print("Unique dates per ATM:")
display(date_count_per_atm.describe())

incomplete_atms = date_count_per_atm[
    date_count_per_atm != df["Date"].nunique()
]

if len(incomplete_atms) == 0:
    print("SUCCESS: Every ATM has observations for every available date.")
else:
    print("WARNING: Some ATMs have missing dates.")
    display(incomplete_atms)


**Check duplicate ATM/date combinations**

In [ ]:
duplicates = df.duplicated(
    subset=["ATMID", "Date"]
).sum()

print("Duplicate ATM-Date combinations:", duplicates)

if duplicates == 0:
    print("SUCCESS: No duplicate ATM-Date records.")
else:
    print("WARNING: Duplicate ATM-Date records detected.")


In [ ]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_table = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (missing / len(df)) * 100
})

display(
    missing_table[
        missing_table["Missing_Count"] > 0
    ]
)


**Rebuild next day target**

In [ ]:
df = (
    df.sort_values(["ATMID", "Date"])
      .reset_index(drop=True)
)

# Recalculate next-day withdrawal for each ATM
def future_value(frame, col, days):
    """Value of `col` exactly `days` calendar days later for the same ATM; NaN if that date is
    absent. (groupby.shift would silently pair 25-Mar with 1-Jul because months are missing.)"""
    t = frame[["ATMID", "Date", col]].copy()
    t["Date"] = t["Date"] - pd.Timedelta(days=days)
    return frame[["ATMID", "Date"]].merge(t, on=["ATMID", "Date"], how="left")[col].to_numpy()

df["Target_Next_Day_Calculated"] = future_value(df, "Daily_Withdrawal", 1)

# Compare existing target with calculated target
comparison_mask = (
    df["Target_Next_Day"].notna() &
    df["Target_Next_Day_Calculated"].notna()
)

target_difference = (
    df.loc[comparison_mask, "Target_Next_Day"]
    -
    df.loc[comparison_mask, "Target_Next_Day_Calculated"]
).abs()

print("Rows compared:", comparison_mask.sum())
print("Maximum target difference:", target_difference.max())
print("Mean target difference:", target_difference.mean())

if target_difference.max() == 0:
    print("\nSUCCESS: Existing target matches the calculated next-day demand.")
else:
    print("\nWARNING: Existing target contains differences.")


In [ ]:
df["Target_Next_Day"] = df["Target_Next_Day_Calculated"]

df.drop(
    columns=["Target_Next_Day_Calculated"],
    inplace=True
)

print("Target successfully reconstructed.")


**Remove rows without a real next-day target**

In [ ]:
df_full = df.copy()   # keeps the latest date: it has no next-day target but is the true forecast origin
before_rows = len(df)

df = df.dropna(
    subset=["Target_Next_Day"]
).copy()

after_rows = len(df)

print("Rows before:", before_rows)
print("Rows after :", after_rows)
print("Rows removed:", before_rows - after_rows)

print("\nFinal forecasting date range:")
print(df["Date"].min(), "to", df["Date"].max())


**Check target distribution**

In [ ]:
print("Target statistics:")

display(
    df["Target_Next_Day"]
      .describe()
      .to_frame()
)

plt.figure(figsize=(12, 6))

sns.histplot(
    df["Target_Next_Day"],
    bins=50,
    kde=True
)

plt.title("Distribution of Next-Day ATM Cash Demand")
plt.xlabel("Next-Day Withdrawal")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


**Overall daily demand**

In [ ]:
daily_demand = (
    df.groupby("Date")["Daily_Withdrawal"]
      .sum()
      .reset_index()
)

plt.figure(figsize=(15, 6))

plt.plot(
    daily_demand["Date"],
    daily_demand["Daily_Withdrawal"],
    linewidth=2
)

plt.title("Total Daily ATM Withdrawal Demand")
plt.xlabel("Date")
plt.ylabel("Total Withdrawal Amount")

plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
weekday_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

weekday_demand = (
    df.groupby("DayOfWeek")["Daily_Withdrawal"]
      .mean()
      .reset_index()
)

weekday_demand["Day_Name"] = (
    weekday_demand["DayOfWeek"]
    .map(weekday_names)
)

display(weekday_demand)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=weekday_demand,
    x="Day_Name",
    y="Daily_Withdrawal"
)

plt.title("Average ATM Demand by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Withdrawal")

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


**ATM-level Demand**

In [ ]:
atm_demand = (
    df.groupby("ATMID")["Daily_Withdrawal"]
      .agg(
          Total_Demand="sum",
          Average_Demand="mean",
          Maximum_Demand="max",
          Minimum_Demand="min"
      )
      .sort_values(
          "Average_Demand",
          ascending=False
      )
)

display(atm_demand.head(20))


In [ ]:
feature_cols = [

    # Lag features
    "Withdrawal_Lag_1",
    "Withdrawal_Lag_2",
    "Withdrawal_Lag_3",
    "Withdrawal_Lag_7",
    "Withdrawal_Lag_14",
    "Withdrawal_Lag_21",
    "Withdrawal_Lag_28",

    # Rolling mean
    "Rolling_Mean_3",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_28",

    # Rolling volatility
    "Rolling_Std_7",
    "Rolling_Std_14",
    "Rolling_Std_28",

    # Exponential moving averages
    "EWMA_7",
    "EWMA_14",

    # Demand trend
    "Demand_Change_1D",
    "Demand_Change_7D",
    "Demand_Growth_1D",
    "Demand_Growth_7D",

    # Demand ratio
    "Short_Long_Demand_Ratio",

    # Transaction features
    "Transaction_Count",
    "Average_Transaction",
    "Median_Transaction",
    "Maximum_Transaction",
    "Minimum_Transaction",
    "Withdrawal_Std",

    # Calendar features
    "Day",
    "DayOfWeek",
    "Month",
    "Quarter",
    "WeekOfYear",
    "IsWeekend",
    "IsMonthStart",
    "IsMonthEnd",

    # Historical weekday behavior
    "Weekday_Average_Demand"
]

missing_features = [
    col for col in feature_cols
    if col not in df.columns
]

print("Number of selected features:", len(feature_cols))

if missing_features:
    print("\nMissing features:")
    print(missing_features)
else:
    print("SUCCESS: All selected features exist.")


**Prepare Modeling Dataset**

In [ ]:
model_df = df[
    ["ATMID", "Date"] +
    feature_cols +
    ["Target_Next_Day"]
].copy()

print("Before removing feature NaN values:")
print(model_df.shape)

model_df = model_df.dropna(
    subset=feature_cols + ["Target_Next_Day"]
).copy()

model_df = (
    model_df.sort_values(["Date", "ATMID"])
            .reset_index(drop=True)
)

print("\nAfter removing unusable rows:")
print(model_df.shape)

print(
    "\nRemaining missing values:",
    model_df[feature_cols + ["Target_Next_Day"]]
    .isna()
    .sum()
    .sum()
)


**Modeling Data Range**

In [ ]:
print("Modeling date range:")
print(
    model_df["Date"].min(),
    "→",
    model_df["Date"].max()
)

print(
    "\nNumber of dates:",
    model_df["Date"].nunique()
)

print(
    "Number of ATMs:",
    model_df["ATMID"].nunique()
)

print(
    "Rows:",
    len(model_df)
)


**CHRONOLOGICAL DATE-BASED SPLIT**

In [ ]:
unique_dates = (
    model_df["Date"]
    .sort_values()
    .unique()
)

n_dates = len(unique_dates)

train_date_end = int(n_dates * 0.70)
validation_date_end = int(n_dates * 0.85)

train_dates = unique_dates[:train_date_end]
validation_dates = unique_dates[
    train_date_end:validation_date_end
]
test_dates = unique_dates[
    validation_date_end:
]

train = model_df[
    model_df["Date"].isin(train_dates)
].copy()

validation = model_df[
    model_df["Date"].isin(validation_dates)
].copy()

test = model_df[
    model_df["Date"].isin(test_dates)
].copy()

print("TRAIN")
print(train.shape)
print(train["Date"].min(), "→", train["Date"].max())

print("\nVALIDATION")
print(validation.shape)
print(validation["Date"].min(), "→", validation["Date"].max())

print("\nTEST")
print(test.shape)
print(test["Date"].min(), "→", test["Date"].max())


**TRAIN-ONLY OUTLIER BOUNDS (LEAKAGE-SAFE)**


---


Bounds will be computed from the training split only, so no information from validation/test leaks into how outliers are defined.

In [ ]:
# Compute per-ATM outlier bounds using ONLY the training split (no leakage)
train_target_stats = (
    train.groupby("ATMID")["Target_Next_Day"]
         .agg(Target_Q1=lambda x: x.quantile(0.25),
              Target_Q3=lambda x: x.quantile(0.75))
         .reset_index()
)

train_target_stats["IQR"] = train_target_stats["Target_Q3"] - train_target_stats["Target_Q1"]
train_target_stats["Upper_Bound"] = train_target_stats["Target_Q3"] + 1.5 * train_target_stats["IQR"]

# Fallback for ATMs with IQR == 0
global_upper_fallback = train["Target_Next_Day"].quantile(0.99)
train_target_stats["Upper_Bound"] = train_target_stats["Upper_Bound"].where(
    train_target_stats["IQR"] > 0, global_upper_fallback
)

display(train_target_stats.head(20))


**APPLY CAP TO TRAINING TARGET ONLY**


---


Validation and test keep the real, uncapped demand — only the training target is winsorized, so extreme spike-days can't dominate model training while evaluation still reflects true business error.

In [ ]:
train = train.merge(train_target_stats[["ATMID", "Upper_Bound"]], on="ATMID", how="left")
validation = validation.merge(train_target_stats[["ATMID", "Upper_Bound"]], on="ATMID", how="left")
test = test.merge(train_target_stats[["ATMID", "Upper_Bound"]], on="ATMID", how="left")

train["Upper_Bound"] = train["Upper_Bound"].fillna(global_upper_fallback)
validation["Upper_Bound"] = validation["Upper_Bound"].fillna(global_upper_fallback)
test["Upper_Bound"] = test["Upper_Bound"].fillna(global_upper_fallback)

train["Target_Next_Day_Capped"] = train["Target_Next_Day"].clip(upper=train["Upper_Bound"])

affected = (train["Target_Next_Day_Capped"] < train["Target_Next_Day"]).sum()
print(f"Training rows capped: {affected} ({affected/len(train)*100:.2f}%)")


**VERIFY NO TEMPORAL LEAKAGE**

In [ ]:
print("Train maximum date:")
print(train["Date"].max())

print("\nValidation minimum date:")
print(validation["Date"].min())

print("\nValidation maximum date:")
print(validation["Date"].max())

print("\nTest minimum date:")
print(test["Date"].min())

assert train["Date"].max() < validation["Date"].min()
assert validation["Date"].max() < test["Date"].min()

print("\nSUCCESS: No chronological overlap between datasets.")


In [ ]:
X_train = train[feature_cols].copy()
y_train = train["Target_Next_Day_Capped"].copy()   # capped -> keeps outliers from dominating training

X_val = validation[feature_cols].copy()
y_val = validation["Target_Next_Day"].copy()        # real demand -> honest validation score

X_test = test[feature_cols].copy()
y_test = test["Target_Next_Day"].copy()             # real demand -> honest test score

print("X_train:", X_train.shape, " y_train range:", y_train.min(), "-", y_train.max())
print("X_val:  ", X_val.shape,   " y_val range:  ", y_val.min(), "-", y_val.max())
print("X_test: ", X_test.shape, " y_test range: ", y_test.min(), "-", y_test.max())


**MODEL EVALUATION FUNCTION**

In [ ]:
def evaluate_model(model_name, y_true, y_pred, high_demand_pct=75):
    """
    Regression metrics (MAE, RMSE, MAPE, R2) plus operational
    classification metrics (Precision / Recall / F1) for high-demand days.

    A day is labeled "high demand" when actual demand is at or above the
    high_demand_pct percentile of y_true. The same threshold is applied to
    predictions so we can score how well the model flags high-demand days.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    non_zero = y_true != 0
    if non_zero.sum() > 0:
        mape = (
            np.mean(
                np.abs(
                    (y_true[non_zero] - y_pred[non_zero])
                    / y_true[non_zero]
                )
            ) * 100
        )
    else:
        mape = np.nan

    # Operational classification: high-demand days
    thr = np.percentile(y_true, high_demand_pct)
    y_true_cls = (y_true >= thr).astype(int)
    y_pred_cls = (y_pred >= thr).astype(int)

    precision = precision_score(y_true_cls, y_pred_cls, zero_division=0)
    recall = recall_score(y_true_cls, y_pred_cls, zero_division=0)
    f1 = f1_score(y_true_cls, y_pred_cls, zero_division=0)
    accuracy = accuracy_score(y_true_cls, y_pred_cls)

    result = {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "R2": r2,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy (cls)": accuracy,
        "High_Demand_Threshold": thr,
    }

    print("=" * 60)
    print(model_name)
    print("=" * 60)
    print(f"MAE       : {mae:,.2f}")
    print(f"RMSE      : {rmse:,.2f}")
    print(f"MAPE      : {mape:.2f}%")
    print(f"R²        : {r2:.4f}")
    print(f"Precision : {precision:.4f}  (high-demand @ P{high_demand_pct})")
    print(f"Recall    : {recall:.4f}")
    print(f"F1        : {f1:.4f}")
    print(f"Accuracy  : {accuracy:.4f}  (classification)")
    print(f"Threshold : {thr:,.0f}")

    return result


# LightGBM

In [ ]:
lightgbm_model = LGBMRegressor(
    objective="regression",
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

lightgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
)
print("LightGBM best iteration:", getattr(lightgbm_model, "best_iteration_", lightgbm_model.n_estimators_))


In [ ]:
lgb_val_pred = np.clip(lightgbm_model.predict(X_val), 0, None)
lgb_test_pred = np.clip(lightgbm_model.predict(X_test), 0, None)

lgb_val_result = evaluate_model("LightGBM Validation", y_val, lgb_val_pred)
lgb_test_result = evaluate_model("LightGBM Test", y_test, lgb_test_pred)


In [ ]:
lgb_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": lightgbm_model.feature_importances_
})

lgb_importance = (
    lgb_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(lgb_importance.head(20))

plt.figure(figsize=(10, 8))
top_lgb = lgb_importance.head(20)
plt.barh(top_lgb["Feature"][::-1], top_lgb["Importance"][::-1])
plt.title("Top 20 LightGBM Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


# XGBoost

In [ ]:
xgboost_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=7,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50
)

xgboost_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print("XGBoost best iteration:", getattr(xgboost_model, "best_iteration", xgboost_model.n_estimators))


In [ ]:
xgb_val_pred = np.clip(xgboost_model.predict(X_val), 0, None)
xgb_test_pred = np.clip(xgboost_model.predict(X_test), 0, None)

xgb_val_result = evaluate_model("XGBoost Validation", y_val, xgb_val_pred)
xgb_test_result = evaluate_model("XGBoost Test", y_test, xgb_test_pred)


In [ ]:
xgb_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": xgboost_model.feature_importances_
})

xgb_importance = (
    xgb_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(xgb_importance.head(20))

plt.figure(figsize=(10, 8))
top_xgb = xgb_importance.head(20)
plt.barh(top_xgb["Feature"][::-1], top_xgb["Importance"][::-1])
plt.title("Top 20 XGBoost Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


# CatBoost

In [ ]:
catboost_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    l2_leaf_reg=5,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    early_stopping_rounds=50
)

catboost_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)
print("CatBoost best iteration:", catboost_model.get_best_iteration())


In [ ]:
cat_val_pred = np.clip(catboost_model.predict(X_val), 0, None)
cat_test_pred = np.clip(catboost_model.predict(X_test), 0, None)

cat_val_result = evaluate_model("CatBoost Validation", y_val, cat_val_pred)
cat_test_result = evaluate_model("CatBoost Test", y_test, cat_test_pred)


In [ ]:
cat_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": catboost_model.feature_importances_
})

cat_importance = (
    cat_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(cat_importance.head(20))

plt.figure(figsize=(10, 8))
top_cat = cat_importance.head(20)
plt.barh(top_cat["Feature"][::-1], top_cat["Importance"][::-1])
plt.title("Top 20 CatBoost Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


**MODEL COMPARISON**

In [ ]:
validation_results = pd.DataFrame([lgb_val_result, xgb_val_result, cat_val_result])
test_results = pd.DataFrame([lgb_test_result, xgb_test_result, cat_test_result])

metric_cols = [
    "Model", "MAE", "RMSE", "MAPE (%)", "R2",
    "Precision", "Recall", "F1", "Accuracy (cls)", "High_Demand_Threshold",
]

print("VALIDATION RESULTS (sorted by RMSE)")
display(validation_results[metric_cols].sort_values("RMSE"))

print("\nTEST RESULTS (sorted by RMSE)")
display(test_results[metric_cols].sort_values("RMSE"))

print("\nTEST RESULTS (sorted by F1 — higher is better for high-demand detection)")
display(test_results[metric_cols].sort_values("F1", ascending=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(test_results["Model"], test_results["RMSE"])
ax.set_title("Model Comparison — RMSE")
ax.set_xlabel("Model")
ax.set_ylabel("RMSE")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(test_results["Model"], test_results["MAE"])
plt.title("Model Comparison — MAE")
plt.xlabel("Model")
plt.ylabel("MAE")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(test_results["Model"], test_results["R2"])
plt.title("Model Comparison — R²")
plt.xlabel("Model")
plt.ylabel("R²")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Operational classification metrics (Precision / Recall / F1)**

Because the target is continuous demand, Precision / Recall / F1 are computed by
labeling days as **high demand** when actual demand is at or above the 75th
percentile of the evaluation set. The same threshold is applied to predictions.

- **Precision** — of days predicted high-demand, how many truly were (fewer false refill trips)
- **Recall** — of true high-demand days, how many we caught (fewer stock-outs)
- **F1** — harmonic mean of Precision and Recall

For cash logistics, **recall** is often prioritized over precision.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, metric, title in zip(
    axes,
    ["Precision", "Recall", "F1"],
    ["Precision (High Demand)", "Recall (High Demand)", "F1 (High Demand)"],
):
    ax.bar(test_results["Model"], test_results[metric], color=["#2563eb", "#dc2626", "#059669"])
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=15)
    for i, v in enumerate(test_results[metric]):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


**Best Model**

In [ ]:
# Model selection must use VALIDATION only. The test set is held out for final reporting.
validation_results_sorted = (
    validation_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

best_model_name = validation_results_sorted.loc[0, "Model"]

print("BEST MODEL (selected on validation RMSE only)")
print(best_model_name)

display(validation_results_sorted.iloc[[0]][[
    "Model", "MAE", "RMSE", "MAPE (%)", "R2",
    "Precision", "Recall", "F1", "Accuracy (cls)", "High_Demand_Threshold",
]])

print("\nTEST RESULTS (final unbiased comparison)")
display(test_results.sort_values("RMSE")[[
    "Model", "MAE", "RMSE", "MAPE (%)", "R2",
    "Precision", "Recall", "F1", "Accuracy (cls)", "High_Demand_Threshold",
]])


**Best Model Prediction**

In [ ]:
if best_model_name == "LightGBM Validation":
    best_predictions = lgb_test_pred
    best_model = lightgbm_model
elif best_model_name == "XGBoost Validation":
    best_predictions = xgb_test_pred
    best_model = xgboost_model
elif best_model_name == "CatBoost Validation":
    best_predictions = cat_test_pred
    best_model = catboost_model
else:
    raise ValueError(f"Unknown validation-selected model: {best_model_name}")

# Cash demand cannot be negative operationally.
best_predictions = np.clip(np.asarray(best_predictions, dtype=float), 0, None)

comparison_df = test[["ATMID", "Date", "Target_Next_Day"]].copy()
comparison_df["Predicted_Next_Day"] = best_predictions
comparison_df["Absolute_Error"] = (
    comparison_df["Target_Next_Day"] - comparison_df["Predicted_Next_Day"]
).abs()

display(comparison_df.head(20))


**High-demand confusion matrix (best model, test set)**


In [ ]:
# Confusion matrix for the best model on the test set (high-demand classification)
print("Best model for confusion matrix (selected on validation RMSE):", best_model_name)

# Rebuild high-demand labels using the same rule as evaluate_model
thr = np.percentile(y_test, 75)
y_true_cls = (np.asarray(y_test) >= thr).astype(int)
y_pred_cls = (np.asarray(best_predictions) >= thr).astype(int)

cm = confusion_matrix(y_true_cls, y_pred_cls)
print(f"High-demand threshold (P75 of test actual): {thr:,.0f}")
print("Confusion matrix [TN FP; FN TP]:")
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred Low", "Pred High"],
    yticklabels=["Actual Low", "Actual High"],
    ax=ax,
)
ax.set_title(f"High-Demand Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

print(f"Precision : {precision_score(y_true_cls, y_pred_cls, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_true_cls, y_pred_cls, zero_division=0):.4f}")
print(f"F1        : {f1_score(y_true_cls, y_pred_cls, zero_division=0):.4f}")


In [ ]:
# Full-period predictions for audit/visualization.
X_full = model_df[feature_cols].copy()
full_predictions = np.clip(np.asarray(best_model.predict(X_full), dtype=float), 0, None)

full_comparison_df = model_df[["ATMID", "Date", "Target_Next_Day"]].copy()
full_comparison_df["Predicted_Next_Day"] = full_predictions
full_comparison_df["Absolute_Error"] = (
    full_comparison_df["Target_Next_Day"] - full_comparison_df["Predicted_Next_Day"]
).abs()

full_comparison_df["Split"] = np.select(
    [full_comparison_df["Date"].isin(train_dates),
     full_comparison_df["Date"].isin(validation_dates),
     full_comparison_df["Date"].isin(test_dates)],
    ["Train", "Validation", "Test"],
    default="Unknown"
)

full_comparison_df = (
    full_comparison_df.sort_values(["ATMID", "Date"]).reset_index(drop=True)
)

print("Full comparison date range:", full_comparison_df["Date"].min(), "→", full_comparison_df["Date"].max())
print(full_comparison_df["Split"].value_counts())
display(full_comparison_df.head(20))


**CONSERVATIVE ATM CASH LOADING POLICY**

In [ ]:
full_comparison_df["Predicted_Demand"] = (
    full_comparison_df["Predicted_Next_Day"].clip(lower=0)
)

# Conservative safety factor
SAFETY_FACTOR = 1.05

full_comparison_df["Recommended_Loading"] = (
    full_comparison_df["Predicted_Demand"] * SAFETY_FACTOR
)

print("Safety factor:", SAFETY_FACTOR)
display(full_comparison_df.head(20))


In [ ]:
# Save final feature-engineered dataset
output_file = os.path.join(OUTPUT_DIR, "ATM_Cash_Forecasting_Feature_Engineered.csv")
df.to_csv(output_file, index=False)

# Verify
print(f"File saved: {output_file}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("Columns:", list(df.columns))


**ATM CASH CAPACITY CONSTRAINT**

In [ ]:
ATM_MAX_CAPACITY = ATM_CAPACITY   # 4,000,000 (was wrongly 1,500,000)

full_comparison_df["Recommended_Loading"] = (
    full_comparison_df["Recommended_Loading"].clip(lower=0, upper=ATM_MAX_CAPACITY)
)

print("Maximum allowed loading:", ATM_MAX_CAPACITY)


**CASH LOADING ROUNDING**

In [ ]:
CASH_UNIT = 500

full_comparison_df["Recommended_Loading"] = (
    np.ceil(full_comparison_df["Recommended_Loading"] / CASH_UNIT) * CASH_UNIT
)

assert (full_comparison_df["Recommended_Loading"] <= ATM_MAX_CAPACITY).all() and (full_comparison_df["Recommended_Loading"] % CASH_UNIT == 0).all()
display(
    full_comparison_df[["ATMID", "Date", "Predicted_Demand", "Recommended_Loading"]].head(20)
)


**IDLE CASH CONTROL**

In [ ]:
full_comparison_df["Expected_Excess_Cash"] = (
   full_comparison_df["Recommended_Loading"] - full_comparison_df["Predicted_Demand"]
)

full_comparison_df["Excess_Cash_Percentage"] = np.where(
    full_comparison_df["Predicted_Demand"] > 0,
    (full_comparison_df["Expected_Excess_Cash"] / full_comparison_df["Predicted_Demand"]) * 100,
    0
)

display(
   full_comparison_df[[
        "ATMID", "Date", "Predicted_Demand",
        "Recommended_Loading", "Expected_Excess_Cash", "Excess_Cash_Percentage"
    ]].head(20)
)


In [ ]:
daily_actual_vs_pred = (
    full_comparison_df
    .groupby("Date")[["Target_Next_Day", "Predicted_Next_Day"]]
    .sum()
    .reset_index()
)

plt.figure(figsize=(18, 7))
plt.plot(daily_actual_vs_pred["Date"], daily_actual_vs_pred["Target_Next_Day"], label="Actual")
plt.plot(daily_actual_vs_pred["Date"], daily_actual_vs_pred["Predicted_Next_Day"], label="Predicted")

plt.axvspan(train_dates.min(), train_dates.max(), color="green", alpha=0.05, label="Train")
plt.axvspan(validation_dates.min(), validation_dates.max(), color="orange", alpha=0.08, label="Validation")
plt.axvspan(test_dates.min(), test_dates.max(), color="red", alpha=0.08, label="Test")

plt.title(f"Actual vs Predicted Total ATM Cash Demand — {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Total Cash Demand (all ATMs)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# The predicted cash

In [ ]:
full_comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "ATM_Cash_Forecasting_Full_Year_Predictions.csv"),
    index=False
)
print("Saved: ATM_Cash_Forecasting_Full_Year_Predictions.csv")


**ACTUAL VS PREDICTED**

In [ ]:
plot_df = full_comparison_df.sort_values("Date").reset_index(drop=True)

plt.figure(figsize=(16, 7))
plt.plot(plot_df["Target_Next_Day"].values[:500], label="Actual")
plt.plot(plot_df["Predicted_Next_Day"].values[:500], label="Predicted")

plt.title(f"Actual vs Predicted ATM Cash Demand — {best_model_name}")
plt.xlabel("Test Observations")
plt.ylabel("Cash Demand")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**ATM-LEVEL MODEL PERFORMANCE**

In [ ]:
atm_results = []

for atm_id, group in full_comparison_df.groupby("ATMID"):
    actual = group["Target_Next_Day"].values
    predicted = group["Predicted_Next_Day"].values

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    atm_results.append({
        "ATMID": atm_id,
        "MAE": mae,
        "RMSE": rmse,
        "Average_Actual_Demand": actual.mean(),
        "Average_Predicted_Demand": predicted.mean()
    })

atm_performance = pd.DataFrame(atm_results)
atm_performance = atm_performance.sort_values("RMSE").reset_index(drop=True)

display(atm_performance.head(20))


In [ ]:
# Check the dataset
print("Rows:", len(atm_performance))
print("Columns:", list(atm_performance.columns))

display(atm_performance.head())

# Save locally
file_name_out = os.path.join(OUTPUT_DIR, "ATM_Level_Model_Performance.csv")
atm_performance.to_csv(file_name_out, index=False)
print("Saved:", file_name_out)


**Worst Performing ATMs**

In [ ]:
display(
    atm_performance.sort_values("RMSE", ascending=False).head(20)
)


In [ ]:
test_results.sort_values("RMSE").to_csv(os.path.join(OUTPUT_DIR, "ATM_Model_Comparison.csv"), index=False)
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "ATM_Test_Predictions.csv"), index=False)
atm_performance.to_csv(os.path.join(OUTPUT_DIR, "ATM_Level_Performance.csv"), index=False)


In [ ]:
joblib.dump(best_model, os.path.join(OUTPUT_DIR, "Best_ATM_Cash_Forecasting_Model.pkl"))
print("Best model saved successfully:")
print("Best_ATM_Cash_Forecasting_Model.pkl")


In [ ]:
joblib.dump(feature_cols, os.path.join(OUTPUT_DIR, "ATM_Forecasting_Feature_List.pkl"))
print("Feature list saved successfully.")
print("Number of features:", len(feature_cols))


In [ ]:
forecasting_package = {
    "model": best_model,
    "feature_columns": feature_cols,
    "model_name": best_model_name,
    "training_date_start": train["Date"].min(),
    "training_date_end": train["Date"].max(),
    "validation_date_start": validation["Date"].min(),
    "validation_date_end": validation["Date"].max(),
    "test_date_start": test["Date"].min(),
    "test_date_end": test["Date"].max()
}

joblib.dump(forecasting_package, os.path.join(OUTPUT_DIR, "ATM_Cash_Forecasting_Package.pkl"))
print("Complete forecasting package saved.")


In [ ]:
output_files = [
    "Best_ATM_Cash_Forecasting_Model.pkl",
    "ATM_Forecasting_Feature_List.pkl",
    "ATM_Cash_Forecasting_Package.pkl",
    "ATM_Model_Comparison.csv",
    "ATM_Test_Predictions.csv",
    "ATM_Level_Performance.csv"
]

print("Saved files:")
for file_name_check in output_files:
    full_path = os.path.join(OUTPUT_DIR, file_name_check)
    status = "OK" if os.path.exists(full_path) else "MISSING"
    print(f"  [{status}] {full_path}")


In [ ]:
output_file = os.path.join(OUTPUT_DIR, "ATM_Cash_Forecasting_Feature_Engineered_Full_Year.csv")
df.to_csv(output_file, index=False)

print(f"File saved: {output_file}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("Columns:", list(df.columns))


# ATM Cash Management Dashboard — Full Year 2024

Interactive **enhanced** dashboard built on the combined Jan–Dec 2024 data.  
Includes Days of Cash, multi-filters, sortable table, progress bar, and responsive layout.


No extra installs needed for this part — the dashboard is now a self-contained HTML page (Plotly loads from its CDN inside that page), opened with Python's built-in `webbrowser` module.

**Build a realistic cash-balance simulation per ATM**


In [ ]:
REORDER_THRESHOLD_PCT = 20     # <= this % of capacity -> "Refill Now"
MEDIUM_CASH_THRESHOLD_PCT = 40  # <= this % of capacity -> "Refill Soon"
ATM_CAPACITY = 4_000_000        # fixed capacity of every ATM

def simulate_end_of_day_cash(frame):
    """Estimated CLOSING cash per ATM/date (no balance feed exists in the data).
    Start full, deduct the ACTUAL withdrawal of each day; if closing cash <= reorder point the ATM is
    refilled to capacity next morning. A gap in the dates restarts the ATM at full capacity."""
    floor_ = ATM_CAPACITY * REORDER_THRESHOLD_PCT / 100.0
    frame = frame.sort_values(["ATMID", "Date"]).reset_index(drop=True)
    out = np.zeros(len(frame), dtype=float)
    w = frame["Daily_Withdrawal"].to_numpy(dtype=float)
    d = frame["Date"].to_numpy()
    # Modern, non-private groupby iteration
    for _, group in frame.groupby("ATMID", sort=False):
        idx = group.index.to_numpy()
        start = float(ATM_CAPACITY)
        for k, j in enumerate(idx):
            if k > 0 and (d[j] - d[idx[k - 1]]) > np.timedelta64(1, "D"):
                start = float(ATM_CAPACITY)
            end = max(start - w[j], 0.0)
            out[j] = end
            start = float(ATM_CAPACITY) if end <= floor_ else end
    return frame[["ATMID", "Date"]].assign(Cash_Balance=out)

# True forecast origin = newest date with complete features (it has no target yet, so it is not in
# full_comparison_df). Predict it with the selected 1-day model.
_ok = df_full[feature_cols].notna().all(axis=1)
latest_date = df_full.loc[_ok, "Date"].max()
latest_rows_1d = df_full[_ok & (df_full["Date"] == latest_date)].copy()
latest_rows_1d["Predicted_Next_Day"] = np.clip(best_model.predict(latest_rows_1d[feature_cols]), 0, None)
latest_rows_1d["Predicted_Demand"] = latest_rows_1d["Predicted_Next_Day"]

stale_atms = sorted(set(df_full["ATMID"]) - set(latest_rows_1d["ATMID"]))
if stale_atms:
    print(f"WARNING: {len(stale_atms)} ATM(s) have no usable data on {latest_date.date()} and are excluded: {stale_atms}")

atm_capacity = pd.Series(
    ATM_CAPACITY,
    index=pd.Index(df_full["ATMID"].unique(), name="ATMID"),
    name="ATM_Capacity",
    dtype="float64"
)

cash_sim_df = simulate_end_of_day_cash(df_full)

today_df = (
    latest_rows_1d[["ATMID", "Date", "Predicted_Next_Day", "Predicted_Demand"]]
    .merge(cash_sim_df[cash_sim_df["Date"] == latest_date][["ATMID", "Cash_Balance"]], on="ATMID", how="left")
    .merge(atm_capacity.reset_index(), on="ATMID", how="left")
)
assert today_df["Cash_Balance"].notna().all() and today_df["ATM_Capacity"].notna().all()

today_df["Estimated_Cash_Remaining"] = today_df["Cash_Balance"]
today_df["Cash_Remaining_Pct"] = (
    today_df["Estimated_Cash_Remaining"] / today_df["ATM_Capacity"] * 100
).round(1)

def refill_flag(pct):
    if pct <= REORDER_THRESHOLD_PCT:
        return "Refill Now"
    elif pct <= MEDIUM_CASH_THRESHOLD_PCT:
        return "Refill Soon"
    else:
        return "OK"

today_df["Status"] = today_df["Cash_Remaining_Pct"].apply(refill_flag)

# Top-up to full, rounded DOWN to the cash unit so the ATM can never exceed capacity.
_room = (today_df["ATM_Capacity"] - today_df["Estimated_Cash_Remaining"]).clip(lower=0)
today_df["Refill_Suggestion_Amount"] = np.floor(_room / CASH_UNIT) * CASH_UNIT
assert (today_df["Estimated_Cash_Remaining"] + today_df["Refill_Suggestion_Amount"] <= today_df["ATM_Capacity"] + 1e-9).all()

# Location: use it if the feature dataset has it, otherwise fall back to a readable label
loc_col = next((c for c in df.columns if "location" in c.lower()), None)
if loc_col:
    loc_map = df.drop_duplicates("ATMID").set_index("ATMID")[loc_col]
    today_df["Location"] = today_df["ATMID"].map(loc_map)
else:
    today_df["Location"] = "ATM " + today_df["ATMID"].astype(str) + " (location not in dataset)"

today_df = today_df.sort_values("Cash_Remaining_Pct").reset_index(drop=True)

print("Status as of:", latest_date.date(), "(closing cash, estimated from actual withdrawals)")
print("Status counts:")
print(today_df["Status"].value_counts())
display(today_df[["ATMID", "Location", "ATM_Capacity", "Estimated_Cash_Remaining",
                   "Cash_Remaining_Pct", "Refill_Suggestion_Amount", "Status"]].head(10))


**Build the dashboard as a standalone page and open it in a new tab**

Same two controls as before — a **search box** and the **ATM dropdown** it filters — but instead of rendering inline under this cell, the dashboard is written out as a self-contained `.html` file and opened in a new browser tab, so it has its own window separate from the notebook.

In [ ]:
import json
import webbrowser
import numpy as np

status_colors = {"Refill Now": "#dc2626", "Refill Soon": "#d97706", "OK": "#059669"}
status_tints = {"Refill Now": "#fef2f2", "Refill Soon": "#fffbeb", "OK": "#ecfdf5"}

# ---------- Prepare ATM records with Days of Cash ----------
atm_records = today_df[[
    "ATMID", "Location", "ATM_Capacity", "Estimated_Cash_Remaining",
    "Cash_Remaining_Pct", "Predicted_Demand", "Refill_Suggestion_Amount", "Status"
]].copy()

atm_records["ATMID"] = atm_records["ATMID"].astype(str).str.strip()
atm_records["Location"] = atm_records["Location"].astype(str).str.strip()

# Days of Cash = remaining cash / predicted next-day demand (guard against zero demand)
atm_records["Days_of_Cash"] = (
    atm_records["Estimated_Cash_Remaining"] /
    atm_records["Predicted_Demand"].clip(lower=1e-6)
).round(1)

atm_records = atm_records.round({
    "ATM_Capacity": 0,
    "Estimated_Cash_Remaining": 0,
    "Cash_Remaining_Pct": 1,
    "Predicted_Demand": 0,
    "Refill_Suggestion_Amount": 0,
    "Days_of_Cash": 1,
})

# Robust Status ordering (avoid fragile sort key)
_status_order = {"Refill Now": 0, "Refill Soon": 1, "OK": 2}
atm_records["_status_rank"] = atm_records["Status"].map(_status_order).fillna(99)
atm_records = (
    atm_records
    .sort_values(["_status_rank", "Days_of_Cash", "Cash_Remaining_Pct"])
    .drop(columns=["_status_rank"])
    .reset_index(drop=True)
)

# NaN-safe conversion for JSON
def _nan_to_none(obj):
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if isinstance(obj, dict):
        return {k: _nan_to_none(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_nan_to_none(v) for v in obj]
    return obj

atm_data_json = _nan_to_none(atm_records.to_dict(orient="records"))

# ---------- KPI totals ----------
total_atms = int(atm_records["ATMID"].nunique())
refill_now_count = int((atm_records["Status"] == "Refill Now").sum())
refill_soon_count = int((atm_records["Status"] == "Refill Soon").sum())
total_refill_amount = float(atm_records["Refill_Suggestion_Amount"].sum())
critical_days = int((atm_records["Days_of_Cash"] < 1.0).sum())
avg_days = float(atm_records["Days_of_Cash"].mean()) if len(atm_records) else 0.0
total_capacity = float(atm_records["ATM_Capacity"].sum())
total_remaining = float(atm_records["Estimated_Cash_Remaining"].sum())
util_pct = round(total_remaining / total_capacity * 100, 1) if total_capacity else 0

# ---------- Full actual-vs-predicted history per ATM ----------
history = {}
for atm_id, group in full_comparison_df.sort_values(["ATMID", "Date"]).groupby("ATMID"):
    history[str(atm_id).strip()] = {
        "dates": group["Date"].dt.strftime("%Y-%m-%d").tolist(),
        "actual": [None if (isinstance(v, float) and math.isnan(v)) else round(float(v), 0)
                   for v in group["Target_Next_Day"].tolist()],
        "predicted": [None if (isinstance(v, float) and math.isnan(v)) else round(float(v), 0)
                      for v in group["Predicted_Next_Day"].tolist()],
    }

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>ATM Cash Management Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  :root {
    --navy: #1e3a5f; --navy-light: #2c4a6e; --grey: #6b7280;
    --red: #dc2626; --orange: #d97706; --green: #059669;
    --card-shadow: 0 1px 3px rgba(0,0,0,0.08), 0 1px 2px rgba(0,0,0,0.06);
    --radius: 12px;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
    background: #f0f2f5; color: #1f2937; line-height: 1.5;
    padding: 16px; min-height: 100vh;
  }
  .container { max-width: 1400px; margin: 0 auto; }
  header {
    display: flex; flex-wrap: wrap; justify-content: space-between;
    align-items: center; margin-bottom: 20px; gap: 12px;
  }
  h1 { font-size: 1.6rem; color: var(--navy); font-weight: 700; }
  .last-updated {
    font-size: 0.85rem; color: var(--grey); background: white;
    padding: 6px 12px; border-radius: 20px; box-shadow: var(--card-shadow);
  }
  .kpi-row {
    display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr));
    gap: 12px; margin-bottom: 24px;
  }
  .kpi-card {
    background: white; border-radius: var(--radius); padding: 16px;
    box-shadow: var(--card-shadow); text-align: center; transition: transform 0.15s;
  }
  .kpi-card:hover { transform: translateY(-2px); }
  .kpi-title { color: var(--grey); font-size: 0.72rem; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600; }
  .kpi-value { font-size: 1.7rem; font-weight: 700; margin-top: 4px; line-height: 1.2; }
  .kpi-sub { font-size: 0.75rem; color: var(--grey); margin-top: 2px; }
  .filters-bar {
    background: white; border-radius: var(--radius); padding: 16px;
    box-shadow: var(--card-shadow); margin-bottom: 20px;
    display: flex; flex-wrap: wrap; gap: 12px; align-items: flex-end;
  }
  .filter-group { display: flex; flex-direction: column; gap: 4px; }
  .filter-group label { font-size: 0.75rem; font-weight: 600; color: var(--grey); text-transform: uppercase; }
  .filters-bar input, .filters-bar select {
    font-size: 0.9rem; padding: 8px 12px; border: 1px solid #d1d5db;
    border-radius: 8px; background: #fff; min-width: 140px;
  }
  .filters-bar input:focus, .filters-bar select:focus {
    outline: none; border-color: var(--navy); box-shadow: 0 0 0 3px rgba(30,58,95,0.15);
  }
  .filters-bar input { min-width: 200px; }
  .match-count { font-size: 0.8rem; color: var(--grey); align-self: center; margin-left: auto; }
  .btn {
    padding: 8px 16px; border: none; border-radius: 8px; font-size: 0.85rem;
    font-weight: 600; cursor: pointer; transition: background 0.15s;
  }
  .btn-secondary { background: #e5e7eb; color: #374151; }
  .btn-secondary:hover { background: #d1d5db; }
  .detail-chart-row {
    display: grid; grid-template-columns: 320px 1fr; gap: 16px; margin-bottom: 24px;
  }
  @media (max-width: 900px) { .detail-chart-row { grid-template-columns: 1fr; } }
  .detail-card {
    background: white; border-radius: var(--radius); padding: 20px;
    box-shadow: var(--card-shadow);
  }
  .detail-card .atm-id { font-size: 1.25rem; font-weight: 700; color: var(--navy); }
  .detail-card .atm-loc { color: var(--grey); font-size: 0.9rem; margin-bottom: 10px; }
  .status-badge {
    display: inline-block; color: white; padding: 4px 14px;
    border-radius: 20px; font-weight: 600; font-size: 0.8rem; margin-bottom: 14px;
  }
  .detail-card hr { border: none; border-top: 1px solid #e5e7eb; margin: 12px 0; }
  .detail-metric {
    display: flex; justify-content: space-between; padding: 6px 0;
    font-size: 0.9rem; border-bottom: 1px solid #f3f4f6;
  }
  .detail-metric:last-child { border-bottom: none; }
  .detail-metric .label { color: var(--grey); }
  .detail-metric .value { font-weight: 600; }
  .progress-bar-wrap {
    background: #e5e7eb; border-radius: 6px; height: 10px; margin: 8px 0 4px; overflow: hidden;
  }
  .progress-bar { height: 100%; border-radius: 6px; transition: width 0.3s; }
  #chart {
    background: white; border-radius: var(--radius);
    box-shadow: var(--card-shadow); padding: 8px; min-height: 380px;
  }
  .table-section {
    background: white; border-radius: var(--radius); box-shadow: var(--card-shadow);
    overflow: hidden; margin-bottom: 20px;
  }
  .table-header {
    display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center;
    padding: 14px 18px; border-bottom: 1px solid #e5e7eb; gap: 10px;
  }
  .table-header h3 { font-size: 1rem; color: var(--navy); }
  .table-wrap { overflow-x: auto; max-height: 520px; overflow-y: auto; }
  table { border-collapse: collapse; width: 100%; font-size: 0.85rem; }
  th {
    background: var(--navy); color: white; padding: 10px 8px; text-align: center;
    position: sticky; top: 0; z-index: 10; cursor: pointer; user-select: none; white-space: nowrap;
  }
  th:hover { background: var(--navy-light); }
  th .sort-icon { opacity: 0.5; font-size: 0.7rem; margin-left: 3px; }
  th.sorted .sort-icon { opacity: 1; }
  td { padding: 8px; text-align: center; border-bottom: 1px solid #f3f4f6; }
  tr:hover td { background: #f8fafc !important; }
  tr.selected-row td {
    border-top: 2px solid var(--navy); border-bottom: 2px solid var(--navy);
    font-weight: 600; background: #eef2ff !important;
  }
  .status-pill {
    display: inline-block; padding: 2px 10px; border-radius: 12px;
    font-size: 0.75rem; font-weight: 600; color: white;
  }
  .pct-cell { font-weight: 600; }
  .days-low { color: var(--red); font-weight: 700; }
  .days-med { color: var(--orange); font-weight: 600; }
  .days-ok { color: var(--green); }
  .footnote { text-align: center; color: var(--grey); font-size: 0.8rem; padding: 12px; }
  .empty-state { padding: 40px; text-align: center; color: var(--grey); }
  @media (max-width: 600px) {
    body { padding: 10px; }
    h1 { font-size: 1.3rem; }
    .kpi-value { font-size: 1.4rem; }
    .filters-bar { flex-direction: column; align-items: stretch; }
    .match-count { margin-left: 0; }
  }
</style>
</head>
<body>
<div class="container">
  <header>
    <h1>ATM Cash Management Dashboard</h1>
    <div class="last-updated">Data as of <strong>__LATEST_DATE__</strong></div>
  </header>

  <div class="kpi-row">
    <div class="kpi-card">
      <div class="kpi-title">Total ATMs</div>
      <div class="kpi-value" style="color:var(--navy);">__TOTAL_ATMS__</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-title">Refill Now</div>
      <div class="kpi-value" style="color:var(--red);">__REFILL_NOW__</div>
      <div class="kpi-sub">Critical</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-title">Refill Soon</div>
      <div class="kpi-value" style="color:var(--orange);">__REFILL_SOON__</div>
      <div class="kpi-sub">Action needed</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-title">&lt; 1 Day Cash</div>
      <div class="kpi-value" style="color:var(--red);">__CRITICAL_DAYS__</div>
      <div class="kpi-sub">High urgency</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-title">Total Refill Needed</div>
      <div class="kpi-value" style="color:var(--green); font-size:1.3rem;">__TOTAL_REFILL__</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-title">Avg Days of Cash</div>
      <div class="kpi-value" style="color:var(--navy);">__AVG_DAYS__</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-title">Network Utilization</div>
      <div class="kpi-value" style="color:var(--navy);">__UTIL_PCT__%</div>
      <div class="kpi-sub">Cash remaining</div>
    </div>
  </div>

  <div class="filters-bar">
    <div class="filter-group">
      <label for="search">Search</label>
      <input id="search" type="text" placeholder="ATM ID or location...">
    </div>
    <div class="filter-group">
      <label for="statusFilter">Status</label>
      <select id="statusFilter">
        <option value="all">All Statuses</option>
        <option value="Refill Now">Refill Now</option>
        <option value="Refill Soon">Refill Soon</option>
        <option value="OK">OK</option>
      </select>
    </div>
    <div class="filter-group">
      <label for="daysFilter">Days of Cash</label>
      <select id="daysFilter">
        <option value="all">Any</option>
        <option value="lt1">&lt; 1 day</option>
        <option value="lt2">&lt; 2 days</option>
        <option value="lt3">&lt; 3 days</option>
        <option value="gte3">&ge; 3 days</option>
      </select>
    </div>
    <div class="filter-group">
      <label for="pctFilter">Remaining %</label>
      <select id="pctFilter">
        <option value="all">Any</option>
        <option value="lt20">&le; 20%</option>
        <option value="lt40">&le; 40%</option>
        <option value="gt40">&gt; 40%</option>
      </select>
    </div>
    <div class="filter-group">
      <label for="atmSelect">Select ATM</label>
      <select id="atmSelect" style="min-width:220px;"></select>
    </div>
    <button class="btn btn-secondary" id="resetFilters" type="button">Reset</button>
    <div class="match-count" id="matchCount"></div>
  </div>

  <div class="detail-chart-row">
    <div class="detail-card" id="detailCard"></div>
    <div id="chart"></div>
  </div>

  <div class="table-section">
    <div class="table-header">
      <h3>All ATMs — Status Overview</h3>
      <span style="font-size:0.8rem;color:var(--grey);">Click column headers to sort · Click row to select</span>
    </div>
    <div class="table-wrap">
      <table id="statusTable">
        <thead>
          <tr>
            <th data-sort="ATMID">ATMID <span class="sort-icon">↕</span></th>
            <th data-sort="Location">Location <span class="sort-icon">↕</span></th>
            <th data-sort="ATM_Capacity">Capacity <span class="sort-icon">↕</span></th>
            <th data-sort="Estimated_Cash_Remaining">Cash Remaining <span class="sort-icon">↕</span></th>
            <th data-sort="Cash_Remaining_Pct">Remaining % <span class="sort-icon">↕</span></th>
            <th data-sort="Days_of_Cash">Days of Cash <span class="sort-icon">↕</span></th>
            <th data-sort="Predicted_Demand">Pred. Demand <span class="sort-icon">↕</span></th>
            <th data-sort="Refill_Suggestion_Amount">Refill Sugg. <span class="sort-icon">↕</span></th>
            <th data-sort="Status">Status <span class="sort-icon">↕</span></th>
          </tr>
        </thead>
        <tbody id="statusTableBody"></tbody>
      </table>
    </div>
  </div>

  <div class="footnote">
    Thresholds: &le;20% = Refill Now · &le;40% = Refill Soon · Days of Cash = Cash Remaining ÷ Predicted Next-Day Demand.
  </div>
</div>

<script>
const atmData = __ATM_DATA_JSON__;
const history = __HISTORY_JSON__;
const statusColors = {"Refill Now": "#dc2626", "Refill Soon": "#d97706", "OK": "#059669"};
const statusTints = {"Refill Now": "#fef2f2", "Refill Soon": "#fffbeb", "OK": "#ecfdf5"};

let selectedId = atmData.length ? String(atmData[0].ATMID) : null;
let currentSort = {key: "Days_of_Cash", asc: true};
let filteredCache = [];

function fmt(n) {
  if (n == null || isNaN(n)) return "—";
  return Math.round(Number(n)).toLocaleString();
}
function fmtPct(n) {
  if (n == null || isNaN(n)) return "—";
  return Number(n).toFixed(1) + "%";
}
function fmtDays(n) {
  if (n == null || isNaN(n)) return "—";
  return Number(n).toFixed(1);
}

function getFiltered() {
  const q = document.getElementById("search").value.trim().toLowerCase();
  const status = document.getElementById("statusFilter").value;
  const daysF = document.getElementById("daysFilter").value;
  const pctF = document.getElementById("pctFilter").value;

  return atmData.filter(r => {
    if (q) {
      const match = String(r.ATMID).toLowerCase().includes(q) ||
                    String(r.Location).toLowerCase().includes(q);
      if (!match) return false;
    }
    if (status !== "all" && r.Status !== status) return false;
    const days = Number(r.Days_of_Cash) || 0;
    if (daysF === "lt1" && !(days < 1)) return false;
    if (daysF === "lt2" && !(days < 2)) return false;
    if (daysF === "lt3" && !(days < 3)) return false;
    if (daysF === "gte3" && !(days >= 3)) return false;
    const pct = Number(r.Cash_Remaining_Pct) || 0;
    if (pctF === "lt20" && !(pct <= 20)) return false;
    if (pctF === "lt40" && !(pct <= 40)) return false;
    if (pctF === "gt40" && !(pct > 40)) return false;
    return true;
  });
}

function sortData(data) {
  const {key, asc} = currentSort;
  const sorted = [...data].sort((a, b) => {
    let va = a[key], vb = b[key];
    if (key === "Status") {
      const order = {"Refill Now": 0, "Refill Soon": 1, "OK": 2};
      va = order[va] ?? 3; vb = order[vb] ?? 3;
    } else if (typeof va === "string") {
      va = va.toLowerCase(); vb = String(vb).toLowerCase();
    } else {
      va = Number(va) || 0; vb = Number(vb) || 0;
    }
    if (va < vb) return asc ? -1 : 1;
    if (va > vb) return asc ? 1 : -1;
    return 0;
  });
  return sorted;
}

function refreshOptions() {
  filteredCache = getFiltered();
  const sorted = sortData(filteredCache);

  document.getElementById("matchCount").textContent =
    `Showing ${sorted.length} of ${atmData.length} ATMs`;

  const select = document.getElementById("atmSelect");
  select.innerHTML = "";
  sorted.forEach(r => {
    const opt = document.createElement("option");
    opt.value = String(r.ATMID);
    opt.textContent = `${r.ATMID} — ${r.Status} (${fmtDays(r.Days_of_Cash)}d)`;
    select.appendChild(opt);
  });

  if (sorted.length) {
    const stillPresent = sorted.some(r => String(r.ATMID) === selectedId);
    selectedId = stillPresent ? selectedId : String(sorted[0].ATMID);
    select.value = selectedId;
  } else {
    selectedId = null;
  }

  document.querySelectorAll("th[data-sort]").forEach(th => {
    th.classList.toggle("sorted", th.dataset.sort === currentSort.key);
    const icon = th.querySelector(".sort-icon");
    if (th.dataset.sort === currentSort.key) {
      icon.textContent = currentSort.asc ? "↑" : "↓";
    } else {
      icon.textContent = "↕";
    }
  });

  renderDetail();
  renderTable(sorted);
}

function renderDetail() {
  const detailCard = document.getElementById("detailCard");
  if (!selectedId) {
    detailCard.innerHTML = "<div class='empty-state'>No ATM matches the current filters.</div>";
    Plotly.purge("chart");
    return;
  }
  const row = atmData.find(r => String(r.ATMID) === selectedId);
  if (!row) {
    detailCard.innerHTML = "<div class='empty-state'>ATM not found.</div>";
    Plotly.purge("chart");
    return;
  }
  const color = statusColors[row.Status] || "#6b7280";
  const pct = Number(row.Cash_Remaining_Pct) || 0;
  const days = Number(row.Days_of_Cash) || 0;
  let barColor = color;
  if (pct > 40) barColor = statusColors["OK"];

  detailCard.innerHTML = `
    <div class="atm-id">ATM ${row.ATMID}</div>
    <div class="atm-loc">${row.Location}</div>
    <div class="status-badge" style="background:${color};">${row.Status}</div>
    <div class="progress-bar-wrap">
      <div class="progress-bar" style="width:${Math.min(pct,100)}%;background:${barColor};"></div>
    </div>
    <div style="font-size:0.8rem;color:var(--grey);margin-bottom:10px;">${fmtPct(pct)} of capacity remaining</div>
    <hr>
    <div class="detail-metric"><span class="label">Vault Capacity</span><span class="value">${fmt(row.ATM_Capacity)}</span></div>
    <div class="detail-metric"><span class="label">Cash Remaining</span><span class="value">${fmt(row.Estimated_Cash_Remaining)}</span></div>
    <div class="detail-metric"><span class="label">Days of Cash</span><span class="value" style="color:${days < 1 ? 'var(--red)' : days < 2 ? 'var(--orange)' : 'var(--green)'}">${fmtDays(days)} days</span></div>
    <div class="detail-metric"><span class="label">Predicted Next-Day Demand</span><span class="value">${fmt(row.Predicted_Demand)}</span></div>
    <div class="detail-metric"><span class="label">Refill Suggestion</span><span class="value">${fmt(row.Refill_Suggestion_Amount)}</span></div>
  `;

  const h = history[selectedId] || {dates: [], actual: [], predicted: []};
  if (!h.dates || h.dates.length === 0) {
    Plotly.purge("chart");
    document.getElementById("chart").innerHTML = "<div class='empty-state'>No historical data available for this ATM.</div>";
    return;
  }
  const traces = [
    { x: h.dates, y: h.actual, name: "Actual Demand", type: "scatter", mode: "lines", line: {color: "#2563eb", width: 2} },
    { x: h.dates, y: h.predicted, name: "Predicted Demand", type: "scatter", mode: "lines", line: {color: "#dc2626", width: 2, dash: "dot"} },
  ];
  Plotly.newPlot("chart", traces, {
    title: {text: `Actual vs Predicted Demand — ATM ${selectedId}`, font: {size: 14}},
    template: "plotly_white",
    xaxis: { title: "Date", gridcolor: "#f0f0f0" },
    yaxis: { title: "Cash Demand", gridcolor: "#f0f0f0" },
    legend: { orientation: "h", y: 1.12 },
    margin: { t: 50, b: 50, l: 60, r: 20 },
    height: 380,
    hovermode: "x unified",
  }, { responsive: true, displayModeBar: true });
}

function renderTable(sorted) {
  const body = document.getElementById("statusTableBody");
  body.innerHTML = "";
  if (!sorted.length) {
    body.innerHTML = "<tr><td colspan='9' class='empty-state'>No ATMs match the current filters.</td></tr>";
    return;
  }
  sorted.forEach(r => {
    const tr = document.createElement("tr");
    tr.style.backgroundColor = statusTints[r.Status] || "white";
    if (selectedId && String(r.ATMID) === selectedId) tr.classList.add("selected-row");

    const days = Number(r.Days_of_Cash) || 0;
    let daysClass = "days-ok";
    if (days < 1) daysClass = "days-low";
    else if (days < 2) daysClass = "days-med";

    const statusColor = statusColors[r.Status] || "#6b7280";

    tr.innerHTML = `
      <td>${r.ATMID}</td>
      <td style="text-align:left;max-width:180px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;" title="${r.Location}">${r.Location}</td>
      <td>${fmt(r.ATM_Capacity)}</td>
      <td>${fmt(r.Estimated_Cash_Remaining)}</td>
      <td class="pct-cell">${fmtPct(r.Cash_Remaining_Pct)}</td>
      <td class="${daysClass}">${fmtDays(days)}</td>
      <td>${fmt(r.Predicted_Demand)}</td>
      <td>${fmt(r.Refill_Suggestion_Amount)}</td>
      <td><span class="status-pill" style="background:${statusColor}">${r.Status}</span></td>
    `;
    tr.addEventListener("click", () => {
      selectedId = String(r.ATMID);
      document.getElementById("atmSelect").value = selectedId;
      renderDetail();
      renderTable(sortData(filteredCache));
    });
    body.appendChild(tr);
  });
}

document.getElementById("search").addEventListener("input", refreshOptions);
document.getElementById("statusFilter").addEventListener("change", refreshOptions);
document.getElementById("daysFilter").addEventListener("change", refreshOptions);
document.getElementById("pctFilter").addEventListener("change", refreshOptions);
document.getElementById("atmSelect").addEventListener("change", (e) => {
  selectedId = e.target.value;
  renderDetail();
  renderTable(sortData(filteredCache));
});
document.getElementById("resetFilters").addEventListener("click", () => {
  document.getElementById("search").value = "";
  document.getElementById("statusFilter").value = "all";
  document.getElementById("daysFilter").value = "all";
  document.getElementById("pctFilter").value = "all";
  currentSort = {key: "Days_of_Cash", asc: true};
  refreshOptions();
});

document.querySelectorAll("th[data-sort]").forEach(th => {
  th.addEventListener("click", () => {
    const key = th.dataset.sort;
    if (currentSort.key === key) {
      currentSort.asc = !currentSort.asc;
    } else {
      currentSort = {key, asc: key === "Days_of_Cash" || key === "Cash_Remaining_Pct" || key === "Status"};
    }
    refreshOptions();
  });
});

refreshOptions();
</script>
</body>
</html>
"""

html_out = (
    HTML_TEMPLATE
    .replace("__TOTAL_ATMS__", str(total_atms))
    .replace("__REFILL_NOW__", str(refill_now_count))
    .replace("__REFILL_SOON__", str(refill_soon_count))
    .replace("__CRITICAL_DAYS__", str(critical_days))
    .replace("__TOTAL_REFILL__", f"{total_refill_amount:,.0f}")
    .replace("__AVG_DAYS__", f"{avg_days:.1f}")
    .replace("__UTIL_PCT__", str(util_pct))
    .replace("__LATEST_DATE__", str(latest_date.date()))
    .replace("__ATM_DATA_JSON__", json.dumps(atm_data_json))
    .replace("__HISTORY_JSON__", json.dumps(history))
)

dashboard_path = os.path.join(OUTPUT_DIR, "ATM_Cash_Dashboard.html")
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(html_out)

dashboard_url = os.path.abspath(dashboard_path)
print('Open this file in your browser:')
print(os.path.abspath(dashboard_path))

print("Enhanced dashboard saved to:", os.path.abspath(dashboard_path))
print(f"  Total ATMs: {total_atms}")
print(f"  Refill Now: {refill_now_count} | Refill Soon: {refill_soon_count}")
print(f"  < 1 Day Cash: {critical_days} | Avg Days of Cash: {avg_days:.1f}")
print(f"  Network Utilization: {util_pct}%")

import webbrowser
_dash_url = "file://" + os.path.abspath(dashboard_path)
print("Dashboard saved to:", os.path.abspath(dashboard_path))
try:
    webbrowser.open_new_tab(_dash_url)
    print("Opened in your browser.")
except Exception:
    print("Open this file manually in your browser:")
    print(os.path.abspath(dashboard_path))


**Enhanced Dashboard features:**
- **Days of Cash** = Cash Remaining ÷ Predicted Next-Day Demand (color-coded)
- Extra KPIs: &lt; 1 Day Cash count, Avg Days of Cash, Network Utilization %
- Filters: Status, Days of Cash bands, Remaining % bands + search
- Clickable column headers to sort the table
- Progress bar on the detail card
- Responsive layout
- Reset Filters button

Tune `REORDER_THRESHOLD_PCT` and `MEDIUM_CASH_THRESHOLD_PCT` in the simulation cell above if you want the refill signal more or less sensitive.


## Production-safe 2-day forecasting and replenishment layer

This section adds a **direct Day+2 forecast**, validation-only model selection, event-calendar support, uncertainty buffers, and an ATM-level replenishment plan. It avoids using test-set performance to choose the model and never produces negative cash-demand forecasts.


In [ ]:
# -------------------- CONFIGURATION --------------------
EVENT_CALENDAR_FILE = "ATM_Event_Calendar.csv"   # Optional, but recommended for production.
CASH_UNIT = 500
SAFETY_BUFFER_MIN_PCT = 0.10
SAFETY_BUFFER_MAX_PCT = 0.30

# Restore the data INCLUDING the newest day (the 1-day section dropped it for lack of a target).
df = df_full.copy()

event_cols = ["IsHoliday", "IsEid", "IsRamadan", "IsSalaryPeriod"]

if os.path.exists(EVENT_CALENDAR_FILE):
    # Prevent duplicate event columns if this section is re-run in the same notebook.
    df = df.drop(columns=[c for c in event_cols if c in df.columns], errors="ignore")
    events = pd.read_csv(EVENT_CALENDAR_FILE)
    required_event_cols = {"Date", "Event_Name", *event_cols}
    missing_event_cols = required_event_cols - set(events.columns)
    if missing_event_cols:
        raise ValueError(f"Event calendar is missing columns: {sorted(missing_event_cols)}")
    events["Date"] = pd.to_datetime(events["Date"], errors="coerce")
    if events["Date"].isna().any():
        raise ValueError("Event calendar contains invalid dates.")
    events = events.drop_duplicates("Date", keep="last").copy()
    for c in event_cols:
        events[c] = pd.to_numeric(events[c], errors="coerce").fillna(0).clip(0, 1).astype(int)
    df = df.merge(events[["Date"] + event_cols], on="Date", how="left")
    for c in event_cols:
        df[c] = df[c].fillna(0).astype(int)
    print("Event calendar loaded:", EVENT_CALENDAR_FILE)
else:
    for c in event_cols:
        df[c] = 0
    print("WARNING: Event calendar not found. Event features remain zero until supplied.")

df = df.sort_values(["ATMID", "Date"]).reset_index(drop=True)
assert not df.duplicated(["ATMID", "Date"]).any(), "Duplicate ATM/date rows after event merge."
print("Rows:", len(df), "| ATMs:", df["ATMID"].nunique(), "| Date range:", df["Date"].min(), "→", df["Date"].max())


In [ ]:
# -------------------- DIRECT 1-DAY / 2-DAY TARGETS --------------------
df["Target_Day_1"] = future_value(df, "Daily_Withdrawal", 1)   # exactly +1 calendar day
df["Target_Day_2"] = future_value(df, "Daily_Withdrawal", 2)   # exactly +2 calendar days

event_features = event_cols
feature_cols_2d = list(dict.fromkeys(feature_cols + event_features))

model_2d = df[
    ["ATMID", "Date"] + feature_cols_2d + ["Target_Day_1", "Target_Day_2"]
].dropna(
    subset=feature_cols_2d + ["Target_Day_1", "Target_Day_2"]
).sort_values(["Date", "ATMID"]).reset_index(drop=True)

unique_dates_2d = np.sort(model_2d["Date"].unique())
n2 = len(unique_dates_2d)
cut1 = int(n2 * 0.70)
cut2 = int(n2 * 0.85)

train_dates_2d = unique_dates_2d[:cut1]
val_dates_2d = unique_dates_2d[cut1:cut2]
test_dates_2d = unique_dates_2d[cut2:]

train2 = model_2d[model_2d["Date"].isin(train_dates_2d)].copy()
val2 = model_2d[model_2d["Date"].isin(val_dates_2d)].copy()
test2 = model_2d[model_2d["Date"].isin(test_dates_2d)].copy()

assert train2["Date"].max() < val2["Date"].min()
assert val2["Date"].max() < test2["Date"].min()

print("2-day direct target dataset:", model_2d.shape)
print("Train:", train2["Date"].min(), "→", train2["Date"].max(), train2.shape)
print("Validation:", val2["Date"].min(), "→", val2["Date"].max(), val2.shape)
print("Test:", test2["Date"].min(), "→", test2["Date"].max(), test2.shape)


In [ ]:
# -------------------- MODEL BENCHMARK FOR DAY+1 AND DAY+2 --------------------
def make_models():
    return {
        "LightGBM": LGBMRegressor(
            objective="regression", n_estimators=2000, learning_rate=0.02,
            num_leaves=31, min_child_samples=20, subsample=0.8,
            colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, n_jobs=-1
        ),
        "XGBoost": XGBRegressor(
            objective="reg:squarederror", n_estimators=2000, learning_rate=0.02,
            max_depth=7, min_child_weight=5, subsample=0.8,
            colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, n_jobs=-1, tree_method="hist"
        ),
        "CatBoost": CatBoostRegressor(
            iterations=2000, learning_rate=0.03, depth=8,
            loss_function="RMSE", eval_metric="RMSE", l2_leaf_reg=5,
            random_seed=42, verbose=False, allow_writing_files=False
        )
    }

def fit_and_score_horizon(target_col):
    Xtr, ytr = train2[feature_cols_2d], train2[target_col]
    Xv, yv = val2[feature_cols_2d], val2[target_col]
    Xt, yt = test2[feature_cols_2d], test2[target_col]
    fitted, rows = {}, []
    for name, model in make_models().items():
        model.fit(Xtr, ytr)
        vp = np.clip(model.predict(Xv), 0, None)
        tp = np.clip(model.predict(Xt), 0, None)
        rows.append({
            "Model": name,
            "Validation_MAE": mean_absolute_error(yv, vp),
            "Validation_RMSE": np.sqrt(mean_squared_error(yv, vp)),
            "Validation_WAPE_%": 100 * np.abs(yv.to_numpy()-vp).sum() / max(yv.sum(), 1),
            "Test_MAE": mean_absolute_error(yt, tp),
            "Test_RMSE": np.sqrt(mean_squared_error(yt, tp)),
            "Test_WAPE_%": 100 * np.abs(yt.to_numpy()-tp).sum() / max(yt.sum(), 1),
        })
        fitted[name] = model
    result = pd.DataFrame(rows).sort_values("Validation_RMSE").reset_index(drop=True)
    return result, result.loc[0, "Model"], fitted

day1_benchmark, selected_day1_name, day1_models = fit_and_score_horizon("Target_Day_1")
day2_benchmark, selected_day2_name, day2_models = fit_and_score_horizon("Target_Day_2")

print("Selected Day+1 model:", selected_day1_name)
display(day1_benchmark)
print("Selected Day+2 model:", selected_day2_name)
display(day2_benchmark)


In [ ]:
# -------------------- FUTURE 2-DAY FORECAST --------------------
# Forecast origin = newest date with complete features. It needs NO target: day+1 / day+2 are the
# unknown future. (Taking model_2d["Date"].max() would forecast days that are already in the data.)
_ok2 = df[feature_cols_2d].notna().all(axis=1)
latest_forecast_date = df.loc[_ok2, "Date"].max()
latest_rows = df[_ok2 & (df["Date"] == latest_forecast_date)].copy()

stale_atms_2d = sorted(set(df["ATMID"]) - set(latest_rows["ATMID"]))
if stale_atms_2d:
    print(f"WARNING: {len(stale_atms_2d)} ATM(s) have no data on {latest_forecast_date.date()} and CANNOT be forecast: {stale_atms_2d}")
assert latest_rows["ATMID"].is_unique and len(latest_rows) > 0

day1_model = day1_models[selected_day1_name]
day2_model = day2_models[selected_day2_name]

future_plan = latest_rows[["ATMID"]].copy()
future_plan["Forecast_Day_1"] = np.clip(day1_model.predict(latest_rows[feature_cols_2d]), 0, None)
future_plan["Forecast_Day_2"] = np.clip(day2_model.predict(latest_rows[feature_cols_2d]), 0, None)

val_day1_pred = np.clip(day1_model.predict(val2[feature_cols_2d]), 0, None)
val_day2_pred = np.clip(day2_model.predict(val2[feature_cols_2d]), 0, None)

v1 = val2[["ATMID", "Target_Day_1"]].copy()
v1["Abs_Error"] = np.abs(v1["Target_Day_1"].to_numpy() - val_day1_pred)
v2 = val2[["ATMID", "Target_Day_2"]].copy()
v2["Abs_Error"] = np.abs(v2["Target_Day_2"].to_numpy() - val_day2_pred)

b1 = v1.groupby("ATMID")["Abs_Error"].apply(lambda s: np.percentile(s, 90) if len(s) >= 5 else np.nan)
b2 = v2.groupby("ATMID")["Abs_Error"].apply(lambda s: np.percentile(s, 90) if len(s) >= 5 else np.nan)
global_buffer = max(float(np.percentile(v1["Abs_Error"], 90)), float(np.percentile(v2["Abs_Error"], 90)))

future_plan["Validation_Error_Buffer"] = future_plan["ATMID"].map(
    pd.concat([b1, b2], axis=1).max(axis=1)
).fillna(global_buffer)

future_plan["Safety_Buffer"] = future_plan["Validation_Error_Buffer"].clip(
    lower=SAFETY_BUFFER_MIN_PCT * (future_plan["Forecast_Day_1"] + future_plan["Forecast_Day_2"]),
    upper=SAFETY_BUFFER_MAX_PCT * (future_plan["Forecast_Day_1"] + future_plan["Forecast_Day_2"])
)

print("Forecast origin:", latest_forecast_date.date())
display(future_plan.head())


In [ ]:
# -------------------- REPLENISHMENT DECISION ENGINE --------------------
# Fixed ATM capacity policy:
# Every ATM has a maximum cash capacity of exactly ৳4,000,000.
# All subsequent forecasting, cash simulation, buffer, risk and replenishment
# calculations continue using this fixed capacity.

ATM_CAPACITY = 4_000_000

# One fixed capacity for every ATM.
planning_capacity = pd.Series(
    ATM_CAPACITY,
    index=future_plan["ATMID"].dropna().unique(),
    name="Planning_Capacity",
    dtype="float64"
)

# Closing cash per ATM on the forecast origin, simulated from ACTUAL withdrawals
# (same helper as the dashboard cell: start full, refill to full next morning when <= 20%).
_sim = simulate_end_of_day_cash(df[df["Date"] <= latest_forecast_date])
sim_balance = _sim[_sim["Date"] == latest_forecast_date].set_index("ATMID")["Cash_Balance"].to_dict()

future_plan["Estimated_Cash_Now"] = future_plan["ATMID"].map(sim_balance)
future_plan["Planning_Capacity"] = future_plan["ATMID"].map(planning_capacity)

future_plan["Projected_End_Day_1_No_Refill"] = (
    future_plan["Estimated_Cash_Now"] - future_plan["Forecast_Day_1"]
).clip(lower=0)

future_plan["Projected_End_Day_2_No_Refill"] = (
    future_plan["Estimated_Cash_Now"]
    - future_plan["Forecast_Day_1"]
    - future_plan["Forecast_Day_2"]
).clip(lower=0)

future_plan["Capacity_Risk"] = (
    future_plan["Planning_Capacity"]
    < future_plan["Forecast_Day_1"] + future_plan["Forecast_Day_2"] + future_plan["Safety_Buffer"]
)

def decide_replenishment(r):
    c, d1, d2, buf, cap = map(float, [
        r["Estimated_Cash_Now"], r["Forecast_Day_1"], r["Forecast_Day_2"],
        r["Safety_Buffer"], r["Planning_Capacity"]
    ])

    if c < d1:
        action = "CRITICAL — REFILL BEFORE DAY 1"
        target = min(cap, d1 + d2 + buf)
        amount = max(0, target - c)
    elif c - d1 < buf:
        action = "REFILL BEFORE DAY 1"
        target = min(cap, d1 + d2 + buf)
        amount = max(0, target - c)
    elif c - d1 - d2 < buf:
        action = "REFILL ON DAY 2"
        target = min(cap, d2 + buf)
        amount = max(0, target - (c - d1))
    else:
        action, amount = "NO REFILL NEEDED", 0

    # Never load above the fixed ৳4,000,000 capacity.
    amount = min(max(amount, 0), max(cap - c, 0))

    # Round DOWN to the nearest cash unit so the final ATM balance can never
    # exceed the physical capacity.
    amount = math.floor(amount / CASH_UNIT) * CASH_UNIT

    return action, amount

future_plan[["Refill_Action", "Required_Refill"]] = future_plan.apply(
    lambda r: pd.Series(decide_replenishment(r)), axis=1
)

# -------------------- CASH DENOMINATION PLAN (BALANCED MIX) --------------------
# Balanced note mix for every refill:
#   • Target ~35% of the refill VALUE in ৳500 notes (enough small notes for customers)
#   • Remaining ~65% in ৳1,000 notes
#   • Total value always equals Required_Refill exactly (both denoms are multiples of ৳500)
#   • Parity is enforced so the residual after ৳500 notes is always divisible by ৳1,000
#
# Tune TARGET_SHARE_500 (0.20 – 0.50) if you want more/fewer ৳500 notes.

TARGET_SHARE_500 = 0.35   # 35% of refill value as ৳500 notes

def balanced_note_mix(amount, share_500=TARGET_SHARE_500):
    """Return (n_1000, n_500) whose values sum exactly to `amount`.

    amount must be a non-negative multiple of 500.
    """
    amount = int(amount)
    if amount <= 0:
        return 0, 0

    max_k = amount // 500                    # max possible ৳500 notes
    target_k = int(round((amount * share_500) / 500))
    need_odd = (amount % 1000) == 500        # residual must match parity

    k = target_k
    # Force correct parity (even k if amount % 1000 == 0, odd k otherwise)
    if need_odd and (k % 2 == 0):
        k += 1
    elif (not need_odd) and (k % 2 == 1):
        k -= 1

    k = max(0, min(k, max_k))

    # Re-check parity after clamping to [0, max_k]
    if need_odd and (k % 2 == 0):
        k = k + 1 if k < max_k else max(k - 1, 0)
    if (not need_odd) and (k % 2 == 1):
        k = k - 1

    n1000 = (amount - 500 * k) // 1000
    return int(n1000), int(k)

_mix = future_plan["Required_Refill"].apply(lambda a: pd.Series(balanced_note_mix(a)))
future_plan["Notes_1000"] = _mix[0].astype(int)
future_plan["Notes_500"] = _mix[1].astype(int)

future_plan["Denomination_Value_1000"] = future_plan["Notes_1000"] * 1_000
future_plan["Denomination_Value_500"] = future_plan["Notes_500"] * 500
future_plan["Denomination_Total"] = (
    future_plan["Denomination_Value_1000"]
    + future_plan["Denomination_Value_500"]
)

# Share checks (informational)
_total_val = future_plan["Denomination_Total"].replace(0, np.nan)
future_plan["Share_500_Pct"] = (
    100 * future_plan["Denomination_Value_500"] / _total_val
).round(1)

future_plan["Risk"] = np.select(
    [future_plan["Capacity_Risk"],
     future_plan["Refill_Action"].eq("CRITICAL — REFILL BEFORE DAY 1")],
    ["CAPACITY RISK", "CRITICAL"],
    default="NORMAL"
)

future_plan["Cash_Remaining_Pct"] = (
    100 * future_plan["Estimated_Cash_Now"] / future_plan["Planning_Capacity"]
).replace([np.inf, -np.inf], np.nan)

display(future_plan[[
    "ATMID", "Forecast_Day_1", "Forecast_Day_2", "Safety_Buffer",
    "Estimated_Cash_Now", "Projected_End_Day_1_No_Refill",
    "Projected_End_Day_2_No_Refill", "Required_Refill",
    "Refill_Action", "Risk"
]].head(20))


### 💵 ATM Refill Denomination Planning (Balanced Mix)

For every recommended refill the notebook calculates a **balanced mix** of **৳1,000** and **৳500** notes:

- Target **~35% of the refill value** in ৳500 notes (so customers can withdraw smaller amounts)
- Remaining **~65%** in ৳1,000 notes
- Total always equals `Required_Refill` exactly
- Parity is enforced so the residual after ৳500 notes is always divisible by ৳1,000

**Example** (৳200,000 refill, 35% target for ৳500s):

| Notes | Count | Value |
|-------|------:|------:|
| ৳1,000 | 130 | ৳130,000 |
| ৳500   | 140 | ৳70,000 |
| **Total** | | **৳200,000** |

Tune `TARGET_SHARE_500` (e.g. 0.25 – 0.45) in the denomination cell if you want more or fewer ৳500 notes.

Columns: `Notes_1000`, `Notes_500`, `Denomination_Value_1000`, `Denomination_Value_500`, `Denomination_Total`, `Share_500_Pct`.


In [ ]:
# -------------------- FINAL AUDIT CHECKS --------------------
numeric_cols = [
    "Forecast_Day_1", "Forecast_Day_2", "Safety_Buffer",
    "Estimated_Cash_Now", "Planning_Capacity", "Required_Refill"
]

# Capacity policy: every ATM must use exactly ৳4,000,000.
assert (np.abs(future_plan["Planning_Capacity"] - 4_000_000) < 1e-6).all(), "ATM capacity policy violation: capacity must be exactly ৳4,000,000."

assert future_plan["ATMID"].is_unique
assert len(future_plan) + len(stale_atms_2d) == df["ATMID"].nunique(), "ATM accounting mismatch"
assert future_plan.loc[future_plan["Refill_Action"].eq("NO REFILL NEEDED"), "Required_Refill"].eq(0).all()
assert latest_forecast_date == df.loc[df[feature_cols_2d].notna().all(axis=1), "Date"].max()
assert future_plan[numeric_cols].notna().all().all(), "NaN found in production decision fields."
assert (future_plan[numeric_cols] >= 0).all().all(), "Negative operational value found."
assert (future_plan["Required_Refill"] % CASH_UNIT == 0).all()
assert (future_plan["Required_Refill"] <= future_plan["Planning_Capacity"] - future_plan["Estimated_Cash_Now"] + 1e-9).all()
assert (future_plan["Estimated_Cash_Now"] + future_plan["Required_Refill"] <= future_plan["Planning_Capacity"] + 1e-9).all()
assert future_plan["Cash_Remaining_Pct"].between(0, 100, inclusive="both").all()
assert (future_plan["Estimated_Cash_Now"] <= future_plan["Planning_Capacity"] + 1e-9).all()

# Denomination audit: balanced ৳1,000 + ৳500 mix must exactly equal Required_Refill
assert (future_plan["Notes_1000"] >= 0).all()
assert (future_plan["Notes_500"] >= 0).all()
assert (
    future_plan["Denomination_Total"] == future_plan["Required_Refill"]
).all(), "Denomination plan does not equal required refill."
assert (
    future_plan["Denomination_Value_1000"]
    + future_plan["Denomination_Value_500"]
    == future_plan["Required_Refill"]
).all()
# Residual after ৳500 notes must be divisible by ৳1,000
assert (
    (future_plan["Required_Refill"] - future_plan["Notes_500"] * 500) % 1000 == 0
).all()

assert selected_day1_name == day1_benchmark.sort_values("Validation_RMSE").iloc[0]["Model"]
assert selected_day2_name == day2_benchmark.sort_values("Validation_RMSE").iloc[0]["Model"]

print("PASS — unique ATM rows:", future_plan["ATMID"].nunique())
print("PASS — every ATM capacity is exactly ৳4,000,000.")
print("PASS — no negative forecasts or refill amounts.")
print("PASS — refill amounts use ৳500 increments and never exceed capacity.")
print("PASS — balanced ৳1,000/৳500 denomination plan exactly matches every refill amount.")
print("PASS — model selection uses validation RMSE only.")
print("PASS — test metrics remain held out for final reporting.")
print("PASS — future decision fields contain no missing values.")

future_plan.to_csv(os.path.join(OUTPUT_DIR, "ATM_2Day_Forecast_and_Replenishment_Plan.csv"), index=False)
day1_benchmark.to_csv(os.path.join(OUTPUT_DIR, "ATM_Day1_Model_Benchmark.csv"), index=False)
day2_benchmark.to_csv(os.path.join(OUTPUT_DIR, "ATM_Day2_Model_Benchmark.csv"), index=False)

joblib.dump({
    "day1_model": day1_model,
    "day2_model": day2_model,
    "day1_model_name": selected_day1_name,
    "day2_model_name": selected_day2_name,
    "feature_columns": feature_cols_2d,
    "forecast_origin": latest_forecast_date,
    "atm_capacity": ATM_CAPACITY
}, os.path.join(OUTPUT_DIR, "ATM_2Day_Forecasting_Package.pkl"))

print("Saved production outputs.")


### 📊 Updated 2-Day Operational Dashboard

This cell **rebuilds** the interactive HTML dashboard using the production 2-day forecast and replenishment plan (`future_plan`).

**What you will see:**
- Forecast Day+1 and Day+2 demand
- Safety buffer and projected cash without refill
- Required refill amount + ৳1,000 / ৳500 note plan
- Refill action (CRITICAL / BEFORE DAY 1 / ON DAY 2 / NONE)
- Risk flag (CAPACITY RISK / CRITICAL / NORMAL)
- Days of cash based on 2-day demand
- Filters, search, sortable table, and detail card


In [ ]:
# -------------------- 2-DAY OPERATIONAL DASHBOARD --------------------
# Rebuilds the HTML dashboard from future_plan so it reflects the full
# 2-day forecast + replenishment decisions (not the older 1-day view).

import json
import math
import os

# ---- Location labels (reuse earlier logic) ----
loc_col = next((c for c in df.columns if "location" in c.lower()), None)
if loc_col:
    _loc_map = df.drop_duplicates("ATMID").set_index("ATMID")[loc_col].to_dict()
else:
    _loc_map = {}

dash = future_plan.copy()
dash["ATMID"] = dash["ATMID"].astype(str).str.strip()
dash["Location"] = dash["ATMID"].map(
    lambda a: _loc_map.get(a, f"ATM {a} (location not in dataset)")
)

# Days of cash using combined 2-day demand
_two_day_demand = (dash["Forecast_Day_1"] + dash["Forecast_Day_2"]).clip(lower=1e-6)
dash["Days_of_Cash"] = (dash["Estimated_Cash_Now"] / _two_day_demand).round(2)

# Friendly short action label for filters / pills
def _short_action(a):
    a = str(a)
    if "CRITICAL" in a:
        return "CRITICAL"
    if "BEFORE DAY 1" in a:
        return "REFILL DAY 1"
    if "DAY 2" in a:
        return "REFILL DAY 2"
    return "OK"

dash["Action_Short"] = dash["Refill_Action"].map(_short_action)

# Round display columns
for col in [
    "Forecast_Day_1", "Forecast_Day_2", "Safety_Buffer",
    "Estimated_Cash_Now", "Projected_End_Day_1_No_Refill",
    "Projected_End_Day_2_No_Refill", "Required_Refill",
    "Planning_Capacity", "Cash_Remaining_Pct", "Days_of_Cash",
    "Denomination_Value_1000", "Denomination_Value_500",
]:
    if col in dash.columns:
        dash[col] = dash[col].round(0 if col != "Cash_Remaining_Pct" and col != "Days_of_Cash" else 1)

dash["Notes_1000"] = dash["Notes_1000"].astype(int)
dash["Notes_500"] = dash["Notes_500"].astype(int)

# Sort: most urgent first
_action_rank = {"CRITICAL": 0, "REFILL DAY 1": 1, "REFILL DAY 2": 2, "OK": 3}
dash["_rank"] = dash["Action_Short"].map(_action_rank).fillna(9)
dash = dash.sort_values(["_rank", "Days_of_Cash", "Cash_Remaining_Pct"]).drop(columns=["_rank"]).reset_index(drop=True)

# ---- KPIs ----
total_atms = int(dash["ATMID"].nunique())
n_critical = int((dash["Action_Short"] == "CRITICAL").sum())
n_day1 = int((dash["Action_Short"] == "REFILL DAY 1").sum())
n_day2 = int((dash["Action_Short"] == "REFILL DAY 2").sum())
n_ok = int((dash["Action_Short"] == "OK").sum())
total_refill = float(dash["Required_Refill"].sum())
avg_days = float(dash["Days_of_Cash"].mean()) if len(dash) else 0.0
util_pct = round(
    float(dash["Estimated_Cash_Now"].sum()) / float(dash["Planning_Capacity"].sum()) * 100, 1
) if float(dash["Planning_Capacity"].sum()) else 0.0
origin_str = str(latest_forecast_date.date())

# ---- NaN-safe JSON ----
def _nan_to_none(obj):
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if isinstance(obj, dict):
        return {k: _nan_to_none(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_nan_to_none(v) for v in obj]
    return obj

keep_cols = [
    "ATMID", "Location", "Planning_Capacity", "Estimated_Cash_Now",
    "Cash_Remaining_Pct", "Days_of_Cash",
    "Forecast_Day_1", "Forecast_Day_2", "Safety_Buffer",
    "Projected_End_Day_1_No_Refill", "Projected_End_Day_2_No_Refill",
    "Required_Refill", "Notes_1000", "Notes_500",
    "Refill_Action", "Action_Short", "Risk",
]
atm_data_json = _nan_to_none(dash[keep_cols].to_dict(orient="records"))

# Optional history from 1-day comparison if available
history = {}
if "full_comparison_df" in dir() or "full_comparison_df" in globals():
    try:
        for atm_id, group in full_comparison_df.sort_values(["ATMID", "Date"]).groupby("ATMID"):
            history[str(atm_id).strip()] = {
                "dates": group["Date"].dt.strftime("%Y-%m-%d").tolist(),
                "actual": [None if (isinstance(v, float) and math.isnan(v)) else round(float(v), 0)
                           for v in group["Target_Next_Day"].tolist()],
                "predicted": [None if (isinstance(v, float) and math.isnan(v)) else round(float(v), 0)
                              for v in group["Predicted_Next_Day"].tolist()],
            }
    except Exception:
        history = {}

HTML_2D = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>ATM 2-Day Cash & Replenishment Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  :root {
    --navy: #1e3a5f; --navy-light: #2c4a6e; --grey: #6b7280;
    --red: #dc2626; --orange: #d97706; --amber: #b45309; --green: #059669;
    --card-shadow: 0 1px 3px rgba(0,0,0,0.08), 0 1px 2px rgba(0,0,0,0.06);
    --radius: 12px;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
    background: #f0f2f5; color: #1f2937; line-height: 1.5;
    padding: 16px; min-height: 100vh;
  }
  .container { max-width: 1500px; margin: 0 auto; }
  header {
    display: flex; flex-wrap: wrap; justify-content: space-between;
    align-items: center; margin-bottom: 20px; gap: 12px;
  }
  h1 { font-size: 1.55rem; color: var(--navy); font-weight: 700; }
  .last-updated {
    font-size: 0.85rem; color: var(--grey); background: white;
    padding: 6px 12px; border-radius: 20px; box-shadow: var(--card-shadow);
  }
  .kpi-row {
    display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr));
    gap: 12px; margin-bottom: 20px;
  }
  .kpi-card {
    background: white; border-radius: var(--radius); padding: 14px;
    box-shadow: var(--card-shadow); text-align: center;
  }
  .kpi-title { color: var(--grey); font-size: 0.7rem; text-transform: uppercase; letter-spacing: 0.4px; font-weight: 600; }
  .kpi-value { font-size: 1.5rem; font-weight: 700; margin-top: 4px; line-height: 1.2; }
  .kpi-sub { font-size: 0.72rem; color: var(--grey); margin-top: 2px; }
  .filters-bar {
    background: white; border-radius: var(--radius); padding: 14px;
    box-shadow: var(--card-shadow); margin-bottom: 18px;
    display: flex; flex-wrap: wrap; gap: 12px; align-items: flex-end;
  }
  .filter-group { display: flex; flex-direction: column; gap: 4px; }
  .filter-group label { font-size: 0.72rem; font-weight: 600; color: var(--grey); text-transform: uppercase; }
  .filters-bar input, .filters-bar select {
    font-size: 0.88rem; padding: 7px 11px; border: 1px solid #d1d5db;
    border-radius: 8px; background: #fff; min-width: 130px;
  }
  .filters-bar input { min-width: 180px; }
  .btn {
    padding: 8px 14px; border: none; border-radius: 8px; font-size: 0.85rem;
    font-weight: 600; cursor: pointer; background: #e5e7eb; color: #374151;
  }
  .btn:hover { background: #d1d5db; }
  .match-count { font-size: 0.8rem; color: var(--grey); align-self: center; margin-left: auto; }
  .detail-chart-row {
    display: grid; grid-template-columns: 340px 1fr; gap: 16px; margin-bottom: 20px;
  }
  @media (max-width: 960px) { .detail-chart-row { grid-template-columns: 1fr; } }
  .detail-card {
    background: white; border-radius: var(--radius); padding: 18px;
    box-shadow: var(--card-shadow);
  }
  .detail-card .atm-id { font-size: 1.2rem; font-weight: 700; color: var(--navy); }
  .detail-card .atm-loc { color: var(--grey); font-size: 0.88rem; margin-bottom: 8px; }
  .status-badge {
    display: inline-block; color: white; padding: 4px 12px;
    border-radius: 20px; font-weight: 600; font-size: 0.78rem; margin-bottom: 12px;
  }
  .detail-metric {
    display: flex; justify-content: space-between; padding: 5px 0;
    font-size: 0.88rem; border-bottom: 1px solid #f3f4f6;
  }
  .detail-metric .label { color: var(--grey); }
  .detail-metric .value { font-weight: 600; }
  .progress-bar-wrap {
    background: #e5e7eb; border-radius: 6px; height: 10px; margin: 8px 0 4px; overflow: hidden;
  }
  .progress-bar { height: 100%; border-radius: 6px; }
  #chart {
    background: white; border-radius: var(--radius);
    box-shadow: var(--card-shadow); padding: 8px; min-height: 360px;
  }
  .table-section {
    background: white; border-radius: var(--radius); box-shadow: var(--card-shadow);
    overflow: hidden; margin-bottom: 20px;
  }
  .table-header {
    display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center;
    padding: 12px 16px; border-bottom: 1px solid #e5e7eb; gap: 8px;
  }
  .table-header h3 { font-size: 1rem; color: var(--navy); }
  .table-wrap { overflow-x: auto; max-height: 540px; overflow-y: auto; }
  table { border-collapse: collapse; width: 100%; font-size: 0.82rem; }
  th {
    background: var(--navy); color: white; padding: 9px 7px; text-align: center;
    position: sticky; top: 0; z-index: 10; cursor: pointer; user-select: none; white-space: nowrap;
  }
  th:hover { background: var(--navy-light); }
  td { padding: 8px 7px; text-align: center; border-bottom: 1px solid #f3f4f6; }
  tr:hover { background: #f9fafb; }
  tr.selected-row { outline: 2px solid var(--navy); }
  .status-pill {
    display: inline-block; color: white; padding: 3px 10px;
    border-radius: 12px; font-size: 0.72rem; font-weight: 600; white-space: nowrap;
  }
  .days-low { color: var(--red); font-weight: 700; }
  .days-med { color: var(--orange); font-weight: 600; }
  .days-ok { color: var(--green); }
  .pct-cell { font-weight: 600; }
</style>
</head>
<body>
<div class="container">
  <header>
    <h1>ATM 2-Day Cash &amp; Replenishment Dashboard</h1>
    <div class="last-updated">Forecast origin: <strong>__LATEST_DATE__</strong></div>
  </header>

  <div class="kpi-row">
    <div class="kpi-card"><div class="kpi-title">Total ATMs</div><div class="kpi-value" id="kpiTotal">256</div></div>
    <div class="kpi-card"><div class="kpi-title">Critical</div><div class="kpi-value" id="kpiCritical" style="color:var(--red)">10</div><div class="kpi-sub">Refill before Day 1</div></div>
    <div class="kpi-card"><div class="kpi-title">Refill Day 1</div><div class="kpi-value" id="kpiDay1" style="color:var(--orange)">11</div></div>
    <div class="kpi-card"><div class="kpi-title">Refill Day 2</div><div class="kpi-value" id="kpiDay2" style="color:var(--amber)">30</div></div>
    <div class="kpi-card"><div class="kpi-title">OK / No refill</div><div class="kpi-value" id="kpiOk" style="color:var(--green)">205</div></div>
    <div class="kpi-card"><div class="kpi-title">Total refill ৳</div><div class="kpi-value" id="kpiRefill" style="font-size:1.15rem">30,001,000</div></div>
    <div class="kpi-card"><div class="kpi-title">TOTAL CASH ON SYS.</div><div class="kpi-value" id="kpiCashSys" style="font-size:1.05rem">—</div></div>
  </div>

  <div class="filters-bar">
    <div class="filter-group">
      <label>Search ATM / Location</label>
      <input type="text" id="search" placeholder="Type ATM ID or location...">
    </div>
    <div class="filter-group">
      <label>Action</label>
      <select id="actionFilter">
        <option value="all">All actions</option>
        <option value="CRITICAL">Critical</option>
        <option value="REFILL DAY 1">Refill Day 1</option>
        <option value="REFILL DAY 2">Refill Day 2</option>
        <option value="OK">OK / No refill</option>
      </select>
    </div>
    <div class="filter-group">
      <label>Risk</label>
      <select id="riskFilter">
        <option value="all">All risk</option>
        <option value="CRITICAL">Critical</option>
        <option value="NORMAL">Normal</option>
      </select>
    </div>
    <div class="filter-group">
      <label>Days of cash</label>
      <select id="daysFilter">
        <option value="all">All</option>
        <option value="lt1">&lt; 1 day</option>
        <option value="1to2">1 – 2 days</option>
        <option value="gt2">&gt; 2 days</option>
      </select>
    </div>
    <button class="btn" id="resetFilters">Reset filters</button>
    <button class="btn" id="exportCsv" style="background:#1e3a5f;color:white">Export filtered CSV</button>
    <div class="filter-group" style="margin-left:8px">
      <label>Upload features CSV</label>
      <input type="file" id="featuresUpload" accept=".csv,text/csv" style="font-size:0.8rem;padding:4px;min-width:160px">
    </div>
    <span class="match-count" id="matchCount"></span>
    <span id="featuresStatus" style="font-size:0.78rem;color:var(--grey);align-self:center"></span>
  </div>

  <div class="detail-chart-row">
    <div class="detail-card" id="detailCard">
      <div class="atm-id" id="dAtmId">Select an ATM</div>
      <div class="atm-loc" id="dLoc"></div>
      <span class="status-badge" id="dStatus" style="display:none"></span>
      <div class="progress-bar-wrap"><div class="progress-bar" id="dBar" style="width:0%;background:#059669"></div></div>
      <div class="detail-metric"><span class="label">Cash remaining %</span><span class="value" id="dPct">—</span></div>
      <div class="detail-metric"><span class="label">Days of cash (2-day)</span><span class="value" id="dDays">—</span></div>
      <div class="detail-metric"><span class="label">Estimated cash now</span><span class="value" id="dCash">—</span></div>
      <div class="detail-metric"><span class="label">Forecast Day+1</span><span class="value" id="dF1">—</span></div>
      <div class="detail-metric"><span class="label">Forecast Day+2</span><span class="value" id="dF2">—</span></div>
      <div class="detail-metric"><span class="label">Safety buffer</span><span class="value" id="dBuf">—</span></div>
      <div class="detail-metric"><span class="label">Proj. end Day 1 (no refill)</span><span class="value" id="dP1">—</span></div>
      <div class="detail-metric"><span class="label">Proj. end Day 2 (no refill)</span><span class="value" id="dP2">—</span></div>
      <div class="detail-metric"><span class="label">Required refill</span><span class="value" id="dRefill">—</span></div>
      <div class="detail-metric"><span class="label">Day+1 notes ৳1,000 / ৳500</span><span class="value" id="dNotes">—</span></div>
      <div class="detail-metric"><span class="label">Risk</span><span class="value" id="dRisk">—</span></div>
      <div class="detail-metric"><span class="label">Action</span><span class="value" id="dAction">—</span></div>
      <div id="featuresSection" style="display:none;margin-top:12px;border-top:1px solid #e5e7eb;padding-top:10px">
        <div style="font-size:0.78rem;font-weight:600;color:var(--navy);margin-bottom:6px">Uploaded forecast features</div>
        <div id="dFeatures" style="font-size:0.8rem;max-height:160px;overflow-y:auto"></div>
      </div>
    </div>
    <div id="chart"></div>
  </div>

  <div class="table-section">
    <div class="table-header">
      <h3>ATM replenishment plan</h3>
      <div>
        <label style="font-size:0.8rem;color:var(--grey)">Jump to ATM </label>
        <select id="atmSelect" style="font-size:0.85rem;padding:5px 8px;border-radius:6px;border:1px solid #d1d5db"></select>
      </div>
    </div>
    <div class="table-wrap">
      <table>
        <thead>
          <tr>
            <th data-sort="ATMID">ATM ID</th>
            <th data-sort="Location">Location</th>
            <th data-sort="Cash_Remaining_Pct">Cash %</th>
            <th data-sort="Days_of_Cash">Days cash</th>
            <th data-sort="Forecast_Day_1">Day+1</th>
            <th data-sort="Forecast_Day_2">Day+2</th>
            <th data-sort="Required_Refill">Refill ৳</th>
            <th data-sort="Notes_1000" title="Notes for next-day (Day+1) demand">Day+1 ×1000</th>
            <th data-sort="Notes_500" title="Notes for next-day (Day+1) demand">Day+1 ×500</th>
            <th data-sort="Action_Short">Action</th>
            <th data-sort="Risk">Risk</th>
          </tr>
        </thead>
        <tbody id="tableBody"></tbody>
      </table>
    </div>
  </div>
</div>

<script>
const ATM_DATA = __ATM_DATA_JSON__;
const HISTORY = __HISTORY_JSON__;

const actionColors = {
  "CRITICAL": "#dc2626",
  "REFILL DAY 1": "#d97706",
  "REFILL DAY 2": "#b45309",
  "OK": "#059669"
};
const actionTints = {
  "CRITICAL": "#fef2f2",
  "REFILL DAY 1": "#fffbeb",
  "REFILL DAY 2": "#fff7ed",
  "OK": "#ecfdf5"
};
const riskColors = {
  "CRITICAL": "#dc2626",
  "NORMAL": "#6b7280"
};

let selectedId = null;
let currentSort = {key: "Days_of_Cash", asc: true};
let filteredCache = ATM_DATA.slice();

function fmt(n) {
  if (n === null || n === undefined || isNaN(n)) return "—";
  return Number(n).toLocaleString("en-US", {maximumFractionDigits: 0});
}
function fmtPct(n) {
  if (n === null || n === undefined || isNaN(n)) return "—";
  return Number(n).toFixed(1) + "%";
}
function fmtDays(n) {
  if (n === null || n === undefined || isNaN(n)) return "—";
  return Number(n).toFixed(2);
}

/** Notes breakdown for next-day (Day+1) cash demand:
 *  ~65% of value in ৳1,000 notes, remainder in ৳500 notes.
 *  Amount is floored to a multiple of ৳500 so the mix is exact. */
function getNotes(amount) {
  amount = Math.floor((Number(amount) || 0) / 500) * 500;
  if (amount <= 0) return {n1000: 0, n500: 0};
  let n1000 = Math.round(0.65 * amount / 1000);
  let n500 = Math.floor((amount - n1000 * 1000) / 500);
  // Guarantee exact total (handles any residual rounding)
  if (n1000 * 1000 + n500 * 500 !== amount) {
    n1000 = Math.floor(amount / 1000);
    n500 = Math.floor((amount - n1000 * 1000) / 500);
  }
  return {n1000: Math.max(0, n1000), n500: Math.max(0, n500)};
}

// Deep copy of original data for Reset
const ORIGINAL_ATM_DATA = JSON.parse(JSON.stringify(ATM_DATA));

// Pre-compute Day+1 demand notes so sorting & display stay consistent
function recomputeNotes(data) {
  data.forEach(r => {
    const n = getNotes(r.Forecast_Day_1);
    r.Notes_1000 = n.n1000;
    r.Notes_500 = n.n500;
  });
}
recomputeNotes(ATM_DATA);

// Uploaded extra feature columns (optional, kept when full data is loaded)
let FEATURES = {};
let FEATURE_COLS = [];

function parseCSV(text) {
  const lines = text.trim().split(/\r?\n/);
  if (lines.length < 2) return {headers: [], rows: []};
  function splitLine(line) {
    const out = [];
    let cur = "", inQ = false;
    for (let i = 0; i < line.length; i++) {
      const c = line[i];
      if (c === '"') { inQ = !inQ; continue; }
      if (c === ',' && !inQ) { out.push(cur.trim()); cur = ""; continue; }
      cur += c;
    }
    out.push(cur.trim());
    return out;
  }
  const headers = splitLine(lines[0]).map(h => h.replace(/^"|"$/g, ""));
  const rows = [];
  for (let i = 1; i < lines.length; i++) {
    if (!lines[i].trim()) continue;
    const vals = splitLine(lines[i]);
    const obj = {};
    headers.forEach((h, j) => { obj[h] = (vals[j] || "").replace(/^"|"$/g, ""); });
    rows.push(obj);
  }
  return {headers, rows};
}

function num(v) {
  if (v === null || v === undefined || v === "") return 0;
  const n = Number(String(v).replace(/,/g, ""));
  return isNaN(n) ? 0 : n;
}

function updateKPIs(data) {
  const total = data.length;
  let critical = 0, day1 = 0, day2 = 0, ok = 0;
  let totalRefill = 0, totalCash = 0;
  let totalCapacity = 0;
  data.forEach(r => {
    const act = (r.Action_Short || "").toUpperCase();
    if (act === "CRITICAL") critical++;
    else if (act === "REFILL DAY 1") day1++;
    else if (act === "REFILL DAY 2") day2++;
    else ok++;
    totalRefill += num(r.Required_Refill);
    totalCash += num(r.Estimated_Cash_Now);
    totalCapacity += num(r.Planning_Capacity) || 4000000; // fallback if missing
  });
  const utilPct = totalCapacity > 0 ? (totalCash / totalCapacity * 100) : 0;
  document.getElementById("kpiTotal").textContent = total;
  document.getElementById("kpiCritical").textContent = critical;
  document.getElementById("kpiDay1").textContent = day1;
  document.getElementById("kpiDay2").textContent = day2;
  document.getElementById("kpiOk").textContent = ok;
  document.getElementById("kpiRefill").textContent = totalRefill.toLocaleString("en-US");
  document.getElementById("kpiCashSys").textContent = utilPct.toFixed(1) + "%";
}

// Map flexible CSV headers → internal field names

const FIELD_MAP = {
  "atmid": "ATMID", "atm_id": "ATMID", "atm id": "ATMID", "atm": "ATMID",
  "location": "Location", "loc": "Location",
  "planning_capacity": "Planning_Capacity", "capacity": "Planning_Capacity",
  "estimated_cash_now": "Estimated_Cash_Now", "cash_now": "Estimated_Cash_Now", "estimated_cash": "Estimated_Cash_Now",
  "cash_remaining_pct": "Cash_Remaining_Pct", "cash_%": "Cash_Remaining_Pct", "cash_pct": "Cash_Remaining_Pct", "cash %": "Cash_Remaining_Pct",
  "days_of_cash": "Days_of_Cash", "days_cash": "Days_of_Cash", "days": "Days_of_Cash", "days cash": "Days_of_Cash",
  "forecast_day_1": "Forecast_Day_1", "forecast_day+1": "Forecast_Day_1", "day+1": "Forecast_Day_1", "forecast1": "Forecast_Day_1", "day_1": "Forecast_Day_1",
  "forecast_day_2": "Forecast_Day_2", "forecast_day+2": "Forecast_Day_2", "day+2": "Forecast_Day_2", "forecast2": "Forecast_Day_2", "day_2": "Forecast_Day_2",
  "safety_buffer": "Safety_Buffer", "buffer": "Safety_Buffer",
  "projected_end_day_1_no_refill": "Projected_End_Day_1_No_Refill", "proj_end_day1": "Projected_End_Day_1_No_Refill",
  "projected_end_day_2_no_refill": "Projected_End_Day_2_No_Refill", "proj_end_day2": "Projected_End_Day_2_No_Refill",
  "required_refill": "Required_Refill", "refill": "Required_Refill", "refill_tk": "Required_Refill", "refill ৳": "Required_Refill",
  "notes_1000": "Notes_1000", "x1000": "Notes_1000", "×1000": "Notes_1000", "day+1 ×1000": "Notes_1000",
  "notes_500": "Notes_500", "x500": "Notes_500", "×500": "Notes_500", "day+1 ×500": "Notes_500",
  "refill_action": "Refill_Action", "action": "Refill_Action",
  "action_short": "Action_Short",
  "risk": "Risk"
};

const OPERATIONAL_FIELDS = new Set([
  "Forecast_Day_1","Forecast_Day_2","Required_Refill","Estimated_Cash_Now",
  "Cash_Remaining_Pct","Days_of_Cash","Safety_Buffer","Planning_Capacity",
  "Projected_End_Day_1_No_Refill","Projected_End_Day_2_No_Refill",
  "Notes_1000","Notes_500","Refill_Action","Action_Short","Risk"
]);

function normalizeRow(raw) {
  const out = {};
  const lowerToOrig = {};
  Object.keys(raw).forEach(k => { lowerToOrig[k.toLowerCase().trim()] = k; });

  Object.entries(FIELD_MAP).forEach(([alias, field]) => {
    if (lowerToOrig[alias] !== undefined) {
      out[field] = raw[lowerToOrig[alias]];
    }
  });

  // Unmapped columns → extra features
  const mappedAliases = new Set(Object.keys(FIELD_MAP));
  Object.keys(raw).forEach(k => {
    const lk = k.toLowerCase().trim();
    if (!mappedAliases.has(lk)) {
      out["__extra_" + k] = raw[k];
    }
  });

  // Coerce numerics
  ["Planning_Capacity","Estimated_Cash_Now","Cash_Remaining_Pct","Days_of_Cash",
   "Forecast_Day_1","Forecast_Day_2","Safety_Buffer",
   "Projected_End_Day_1_No_Refill","Projected_End_Day_2_No_Refill",
   "Required_Refill","Notes_1000","Notes_500"].forEach(f => {
    if (out[f] !== undefined && out[f] !== "") out[f] = num(out[f]);
  });

  if (out.ATMID !== undefined) out.ATMID = String(out.ATMID).trim();

  return out;
}

function deriveOperationalFields(r) {
  // Defaults
  if (!r.Planning_Capacity || r.Planning_Capacity <= 0) r.Planning_Capacity = 4000000;
  if (r.Estimated_Cash_Now === undefined || r.Estimated_Cash_Now === null) r.Estimated_Cash_Now = 0;
  if (r.Forecast_Day_1 === undefined || r.Forecast_Day_1 === null) r.Forecast_Day_1 = 0;
  if (r.Forecast_Day_2 === undefined || r.Forecast_Day_2 === null) r.Forecast_Day_2 = 0;
  if (r.Safety_Buffer === undefined || r.Safety_Buffer === null) r.Safety_Buffer = 0;
  if (r.Required_Refill === undefined || r.Required_Refill === null) r.Required_Refill = 0;
  if (r.Projected_End_Day_1_No_Refill === undefined) {
    r.Projected_End_Day_1_No_Refill = Math.max(0, r.Estimated_Cash_Now - r.Forecast_Day_1);
  }
  if (r.Projected_End_Day_2_No_Refill === undefined) {
    r.Projected_End_Day_2_No_Refill = Math.max(0, r.Projected_End_Day_1_No_Refill - r.Forecast_Day_2);
  }

  // Cash %
  if (r.Cash_Remaining_Pct === undefined || r.Cash_Remaining_Pct === null || r.Cash_Remaining_Pct === "") {
    r.Cash_Remaining_Pct = r.Planning_Capacity > 0
      ? Math.round((r.Estimated_Cash_Now / r.Planning_Capacity) * 1000) / 10
      : 0;
  }

  // Days of cash (2-day basis)
  if (r.Days_of_Cash === undefined || r.Days_of_Cash === null || r.Days_of_Cash === "") {
    const twoDay = (Number(r.Forecast_Day_1) || 0) + (Number(r.Forecast_Day_2) || 0);
    r.Days_of_Cash = twoDay > 0
      ? Math.round((r.Estimated_Cash_Now / twoDay) * 100) / 100
      : 99;
  }

  // Action_Short
  if (!r.Action_Short) {
    if (r.Refill_Action) {
      const a = String(r.Refill_Action).toUpperCase();
      if (a.includes("CRITICAL")) r.Action_Short = "CRITICAL";
      else if (a.includes("BEFORE DAY 1") || a.includes("DAY 1") || a.includes("REFILL DAY 1")) r.Action_Short = "REFILL DAY 1";
      else if (a.includes("DAY 2") || a.includes("REFILL DAY 2")) r.Action_Short = "REFILL DAY 2";
      else r.Action_Short = "OK";
    } else {
      // Derive from days of cash / projections
      const d = Number(r.Days_of_Cash) || 0;
      const p1 = Number(r.Projected_End_Day_1_No_Refill) || 0;
      if (d < 0.5 || p1 <= 0) r.Action_Short = "CRITICAL";
      else if (d < 1) r.Action_Short = "REFILL DAY 1";
      else if (d <= 2) r.Action_Short = "REFILL DAY 2";
      else r.Action_Short = "OK";
    }
  }
  // Normalize Action_Short values
  const act = String(r.Action_Short).toUpperCase().trim();
  if (act.includes("CRITICAL")) r.Action_Short = "CRITICAL";
  else if (act.includes("DAY 1") || act === "REFILL DAY 1") r.Action_Short = "REFILL DAY 1";
  else if (act.includes("DAY 2") || act === "REFILL DAY 2") r.Action_Short = "REFILL DAY 2";
  else if (act === "OK" || act.includes("NO REFILL")) r.Action_Short = "OK";
  else r.Action_Short = "OK";

  if (!r.Risk) r.Risk = (r.Action_Short === "CRITICAL") ? "CRITICAL" : "NORMAL";
  // Normalize Risk
  const rk = String(r.Risk).toUpperCase().trim();
  if (rk.includes("CRITICAL")) r.Risk = "CRITICAL";
  else r.Risk = "NORMAL";

  if (!r.Location) r.Location = "ATM " + (r.ATMID || "");
  if (!r.Refill_Action) {
    if (r.Action_Short === "CRITICAL") r.Refill_Action = "CRITICAL — REFILL BEFORE DAY 1";
    else if (r.Action_Short === "REFILL DAY 1") r.Refill_Action = "REFILL BEFORE DAY 1";
    else if (r.Action_Short === "REFILL DAY 2") r.Refill_Action = "REFILL ON DAY 2";
    else r.Refill_Action = "NO REFILL NEEDED";
  }

  return r;
}

function loadDataFromRows(rows, headers) {
  // Detect: operational plan CSV vs feature-only CSV
  const sample = rows[0] || {};
  const sampleNorm = normalizeRow(sample);
  const hasOperational = (
    sampleNorm.Forecast_Day_1 !== undefined ||
    sampleNorm.Required_Refill !== undefined ||
    sampleNorm.Estimated_Cash_Now !== undefined ||
    sampleNorm.Action_Short !== undefined ||
    sampleNorm.Days_of_Cash !== undefined ||
    sampleNorm.Cash_Remaining_Pct !== undefined
  );

  const newData = [];
  const extraCols = [];

  rows.forEach(raw => {
    const r = normalizeRow(raw);
    if (!r.ATMID) return;
    Object.keys(r).filter(k => k.startsWith("__extra_")).forEach(k => {
      const col = k.slice(8);
      if (!extraCols.includes(col)) extraCols.push(col);
    });
    newData.push(r);
  });

  if (!newData.length) throw new Error("No valid rows with ATMID found.");

  // Collect features
  FEATURES = {};
  FEATURE_COLS = extraCols.slice();
  newData.forEach(r => {
    const id = String(r.ATMID);
    FEATURES[id] = {};
    extraCols.forEach(c => {
      FEATURES[id][c] = r["__extra_" + c];
      delete r["__extra_" + c];
    });
  });

  if (hasOperational) {
    // FULL system replace with operational plan data
    newData.forEach(r => deriveOperationalFields(r));
    ATM_DATA.length = 0;
    newData.forEach(r => ATM_DATA.push(r));
    recomputeNotes(ATM_DATA);
    updateKPIs(ATM_DATA);
    selectedId = null;
    currentSort = {key: "Days_of_Cash", asc: true};
    document.getElementById("search").value = "";
    document.getElementById("actionFilter").value = "all";
    document.getElementById("riskFilter").value = "all";
    document.getElementById("daysFilter").value = "all";
    refreshOptions();
    document.getElementById("featuresStatus").textContent =
      "Loaded plan: " + ATM_DATA.length + " ATMs" +
      (FEATURE_COLS.length ? " + " + FEATURE_COLS.length + " feature cols" : "");
  } else {
    // Features-only CSV: merge features into existing ATMs, do NOT wipe plan
    let matched = 0;
    newData.forEach(r => {
      const id = String(r.ATMID);
      if (ATM_DATA.some(a => String(a.ATMID) === id)) matched++;
    });
    // Also keep any operational-looking fields if present on a per-row basis
    newData.forEach(r => {
      const existing = ATM_DATA.find(a => String(a.ATMID) === String(r.ATMID));
      if (existing) {
        // Optionally overlay any operational fields that were present
        OPERATIONAL_FIELDS.forEach(f => {
          if (r[f] !== undefined && r[f] !== null && r[f] !== "") {
            existing[f] = r[f];
          }
        });
        deriveOperationalFields(existing);
      }
    });
    recomputeNotes(ATM_DATA);
    updateKPIs(ATM_DATA);
    refreshOptions();
    document.getElementById("featuresStatus").textContent =
      "Features loaded: " + Object.keys(FEATURES).length + " rows, " +
      matched + " matched ATMs, " + FEATURE_COLS.length + " feature cols";
  }
}

function applyFilters() {
  const q = (document.getElementById("search").value || "").toLowerCase().trim();
  const act = document.getElementById("actionFilter").value;
  const risk = document.getElementById("riskFilter").value;
  const days = document.getElementById("daysFilter").value;

  return ATM_DATA.filter(r => {
    if (q) {
      const hay = (String(r.ATMID) + " " + String(r.Location)).toLowerCase();
      if (!hay.includes(q)) return false;
    }
    if (act !== "all" && r.Action_Short !== act) return false;
    if (risk !== "all" && r.Risk !== risk) return false;
    const d = Number(r.Days_of_Cash) || 0;
    if (days === "lt1" && !(d < 1)) return false;
    if (days === "1to2" && !(d >= 1 && d <= 2)) return false;
    if (days === "gt2" && !(d > 2)) return false;
    return true;
  });
}

function sortData(rows) {
  const {key, asc} = currentSort;
  return rows.slice().sort((a, b) => {
    let va = a[key], vb = b[key];
    if (typeof va === "string") va = va.toLowerCase();
    if (typeof vb === "string") vb = vb.toLowerCase();
    if (va == null) va = "";
    if (vb == null) vb = "";
    if (va < vb) return asc ? -1 : 1;
    if (va > vb) return asc ? 1 : -1;
    return 0;
  });
}

function renderTable(sorted) {
  const body = document.getElementById("tableBody");
  body.innerHTML = "";
  document.getElementById("matchCount").textContent = sorted.length + " ATM(s)";
  if (!sorted.length) {
    body.innerHTML = '<tr><td colspan="11" style="padding:20px;color:#6b7280">No ATMs match the current filters.</td></tr>';
    return;
  }
  sorted.forEach(r => {
    const tr = document.createElement("tr");
    tr.style.backgroundColor = actionTints[r.Action_Short] || "white";
    if (selectedId && String(r.ATMID) === selectedId) tr.classList.add("selected-row");

    const days = Number(r.Days_of_Cash) || 0;
    let daysClass = "days-ok";
    if (days < 1) daysClass = "days-low";
    else if (days < 2) daysClass = "days-med";

    const actColor = actionColors[r.Action_Short] || "#6b7280";
    const riskColor = riskColors[r.Risk] || "#6b7280";

    tr.innerHTML = `
      <td>${r.ATMID}</td>
      <td style="text-align:left;max-width:160px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;" title="${r.Location}">${r.Location}</td>
      <td class="pct-cell">${fmtPct(r.Cash_Remaining_Pct)}</td>
      <td class="${daysClass}">${fmtDays(days)}</td>
      <td>${fmt(r.Forecast_Day_1)}</td>
      <td>${fmt(r.Forecast_Day_2)}</td>
      <td>${fmt(r.Required_Refill)}</td>
      <td>${r.Notes_1000}</td>
      <td>${r.Notes_500}</td>
      <td><span class="status-pill" style="background:${actColor}">${r.Action_Short}</span></td>
      <td><span class="status-pill" style="background:${riskColor}">${r.Risk}</span></td>
    `;
    tr.addEventListener("click", () => {
      selectedId = String(r.ATMID);
      document.getElementById("atmSelect").value = selectedId;
      renderDetail();
      renderTable(sortData(filteredCache));
    });
    body.appendChild(tr);
  });
}

function renderDetail() {
  const r = ATM_DATA.find(x => String(x.ATMID) === selectedId);
  if (!r) return;
  document.getElementById("dAtmId").textContent = r.ATMID;
  document.getElementById("dLoc").textContent = r.Location || "";
  const badge = document.getElementById("dStatus");
  badge.style.display = "inline-block";
  badge.style.background = actionColors[r.Action_Short] || "#6b7280";
  badge.textContent = r.Action_Short;

  const pct = Number(r.Cash_Remaining_Pct) || 0;
  const bar = document.getElementById("dBar");
  bar.style.width = Math.min(100, Math.max(0, pct)) + "%";
  bar.style.background = pct <= 20 ? "#dc2626" : (pct <= 40 ? "#d97706" : "#059669");

  document.getElementById("dPct").textContent = fmtPct(r.Cash_Remaining_Pct);
  document.getElementById("dDays").textContent = fmtDays(r.Days_of_Cash);
  document.getElementById("dCash").textContent = "৳ " + fmt(r.Estimated_Cash_Now);
  document.getElementById("dF1").textContent = "৳ " + fmt(r.Forecast_Day_1);
  document.getElementById("dF2").textContent = "৳ " + fmt(r.Forecast_Day_2);
  document.getElementById("dBuf").textContent = "৳ " + fmt(r.Safety_Buffer);
  document.getElementById("dP1").textContent = "৳ " + fmt(r.Projected_End_Day_1_No_Refill);
  document.getElementById("dP2").textContent = "৳ " + fmt(r.Projected_End_Day_2_No_Refill);
  document.getElementById("dRefill").textContent = "৳ " + fmt(r.Required_Refill);
  document.getElementById("dNotes").textContent = r.Notes_1000 + " × ৳1,000  +  " + r.Notes_500 + " × ৳500";
  document.getElementById("dRisk").textContent = r.Risk;
  document.getElementById("dAction").textContent = r.Refill_Action;

  // Show uploaded features if available for this ATM
  const featBox = document.getElementById("featuresSection");
  const featDiv = document.getElementById("dFeatures");
  const feats = FEATURES[String(r.ATMID)];
  if (feats && FEATURE_COLS.length) {
    featBox.style.display = "block";
    let html = "";
    FEATURE_COLS.forEach(col => {
      const v = feats[col];
      html += `<div class="detail-metric" style="border:none;padding:2px 0"><span class="label">${col}</span><span class="value">${v === undefined || v === "" ? "—" : v}</span></div>`;
    });
    featDiv.innerHTML = html;
  } else {
    featBox.style.display = "none";
    featDiv.innerHTML = "";
  }

  // History chart if available
  const hist = HISTORY[selectedId];
  const chartDiv = document.getElementById("chart");
  if (hist && hist.dates && hist.dates.length) {
    const traces = [
      {x: hist.dates, y: hist.actual, name: "Actual", type: "scatter", mode: "lines", line: {color: "#1e3a5f", width: 2}},
      {x: hist.dates, y: hist.predicted, name: "Predicted (1-day model)", type: "scatter", mode: "lines", line: {color: "#d97706", width: 2, dash: "dot"}}
    ];
    Plotly.newPlot(chartDiv, traces, {
      margin: {t: 36, r: 16, b: 48, l: 56},
      title: {text: "History — " + selectedId, font: {size: 14}},
      xaxis: {title: "Date"},
      yaxis: {title: "Withdrawal ৳"},
      legend: {orientation: "h", y: 1.12},
      paper_bgcolor: "white",
      plot_bgcolor: "white"
    }, {responsive: true, displayModeBar: false});
  } else {
    // Bar of Day+1 vs Day+2 forecast for this ATM
    Plotly.newPlot(chartDiv, [{
      x: ["Day +1", "Day +2", "Safety buffer", "Required refill"],
      y: [r.Forecast_Day_1, r.Forecast_Day_2, r.Safety_Buffer, r.Required_Refill],
      type: "bar",
      marker: {color: ["#1e3a5f", "#2c4a6e", "#d97706", "#dc2626"]}
    }], {
      margin: {t: 36, r: 16, b: 48, l: 56},
      title: {text: "2-day plan — " + selectedId, font: {size: 14}},
      yaxis: {title: "Amount ৳"},
      paper_bgcolor: "white",
      plot_bgcolor: "white"
    }, {responsive: true, displayModeBar: false});
  }
}

function refreshOptions() {
  filteredCache = applyFilters();
  const sorted = sortData(filteredCache);
  renderTable(sorted);

  const sel = document.getElementById("atmSelect");
  const prev = sel.value;
  sel.innerHTML = "";
  sorted.forEach(r => {
    const opt = document.createElement("option");
    opt.value = String(r.ATMID);
    opt.textContent = r.ATMID + " — " + (r.Location || "").slice(0, 40);
    sel.appendChild(opt);
  });
  if (prev && sorted.some(r => String(r.ATMID) === prev)) {
    sel.value = prev;
    selectedId = prev;
  } else if (sorted.length) {
    selectedId = String(sorted[0].ATMID);
    sel.value = selectedId;
  }
  renderDetail();
}

document.getElementById("search").addEventListener("input", refreshOptions);
document.getElementById("actionFilter").addEventListener("change", refreshOptions);
document.getElementById("riskFilter").addEventListener("change", refreshOptions);
document.getElementById("daysFilter").addEventListener("change", refreshOptions);
document.getElementById("atmSelect").addEventListener("change", (e) => {
  selectedId = e.target.value;
  renderDetail();
  renderTable(sortData(filteredCache));
});
document.getElementById("resetFilters").addEventListener("click", () => {
  // Full system reset to original embedded data
  ATM_DATA.length = 0;
  ORIGINAL_ATM_DATA.forEach(r => ATM_DATA.push(JSON.parse(JSON.stringify(r))));
  recomputeNotes(ATM_DATA);
  FEATURES = {};
  FEATURE_COLS = [];
  selectedId = null;
  currentSort = {key: "Days_of_Cash", asc: true};
  document.getElementById("search").value = "";
  document.getElementById("actionFilter").value = "all";
  document.getElementById("riskFilter").value = "all";
  document.getElementById("daysFilter").value = "all";
  document.getElementById("featuresUpload").value = "";
  document.getElementById("featuresStatus").textContent = "Reset to default data";
  updateKPIs(ATM_DATA);
  refreshOptions();
});

// Export currently filtered rows as CSV
document.getElementById("exportCsv").addEventListener("click", () => {
  try {
    const rows = sortData(filteredCache && filteredCache.length ? filteredCache : ATM_DATA);
    if (!rows.length) { alert("No ATMs to export."); return; }
    const cols = [
      "ATMID","Location","Cash_Remaining_Pct","Days_of_Cash",
      "Estimated_Cash_Now","Forecast_Day_1","Forecast_Day_2",
      "Safety_Buffer","Projected_End_Day_1_No_Refill","Projected_End_Day_2_No_Refill",
      "Required_Refill","Notes_1000","Notes_500","Action_Short","Risk","Refill_Action"
    ];
    const extra = (typeof FEATURE_COLS !== "undefined" ? FEATURE_COLS : []).filter(c => !cols.includes(c));
    const allCols = cols.concat(extra);
    const escape = (v) => {
      if (v === null || v === undefined) return "";
      const s = String(v);
      if (s.includes(",") || s.includes('"') || s.includes("\n") || s.includes("\r")) {
        return '"' + s.replace(/"/g, '""') + '"';
      }
      return s;
    };
    let csv = allCols.join(",") + "\r\n";
    rows.forEach(r => {
      const line = allCols.map(c => {
        if (extra.includes(c)) {
          const f = (typeof FEATURES !== "undefined") ? FEATURES[String(r.ATMID)] : null;
          return escape(f ? f[c] : "");
        }
        return escape(r[c]);
      });
      csv += line.join(",") + "\r\n";
    });
    // BOM helps Excel open UTF-8 correctly
    const blob = new Blob(["\uFEFF" + csv], {type: "text/csv;charset=utf-8;"});
    const url = URL.createObjectURL(blob);
    const a = document.createElement("a");
    a.href = url;
    a.download = "ATM_replenishment_filtered_" + new Date().toISOString().slice(0,10) + ".csv";
    a.style.display = "none";
    document.body.appendChild(a);
    a.click();
    setTimeout(() => {
      document.body.removeChild(a);
      URL.revokeObjectURL(url);
    }, 200);
  } catch (err) {
    console.error("Export failed:", err);
    alert("Export failed: " + err.message);
  }
});

// Upload CSV → replace whole system data
document.getElementById("featuresUpload").addEventListener("change", (e) => {
  const file = e.target.files[0];
  if (!file) return;
  const reader = new FileReader();
  reader.onload = (ev) => {
    try {
      const {headers, rows} = parseCSV(ev.target.result);
      const hasId = headers.some(h => {
        const l = h.toLowerCase().trim();
        return l === "atmid" || l === "atm_id" || l === "atm id" || l === "atm";
      });
      if (!hasId) {
        alert("CSV must contain an ATMID (or ATM_ID) column.");
        return;
      }
      loadDataFromRows(rows, headers);
    } catch (err) {
      alert("Failed to load CSV: " + err.message);
      console.error(err);
    }
  };
  reader.readAsText(file);
});
document.querySelectorAll("th[data-sort]").forEach(th => {
  th.addEventListener("click", () => {
    const key = th.dataset.sort;
    if (currentSort.key === key) currentSort.asc = !currentSort.asc;
    else currentSort = {key, asc: key === "Days_of_Cash" || key === "Cash_Remaining_Pct"};
    refreshOptions();
  });
});

updateKPIs(ATM_DATA);
refreshOptions();
</script>
</body>
</html>
"""

html_out = (
    HTML_2D
    .replace("__LATEST_DATE__", origin_str)
    .replace("__ATM_DATA_JSON__", json.dumps(atm_data_json))
    .replace("__HISTORY_JSON__", json.dumps(history))
)

dashboard_path = os.path.join(OUTPUT_DIR, "ATM_2Day_Cash_Dashboard.html")
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(html_out)

print("2-Day dashboard saved to:", os.path.abspath(dashboard_path))
print(f"  Total ATMs: {total_atms}")
print(f"  Critical: {n_critical} | Refill Day 1: {n_day1} | Refill Day 2: {n_day2} | OK: {n_ok}")
print(f"  Total refill: ৳{total_refill:,.0f}")
print(f"  Network util (TOTAL CASH ON SYS): {util_pct}%")

# Try open in browser
_url = "file://" + os.path.abspath(dashboard_path)
print("Dashboard saved to:", os.path.abspath(dashboard_path))
try:
    webbrowser.open_new_tab(_url)
    print("Opened in your browser.")
except Exception:
    print("Open this file manually in your browser:")
